In [1]:
import numpy as np
import pandas as pd
import re
import requests
import time
import unicodedata
from collections import defaultdict, Counter
from difflib import SequenceMatcher
from sklearn.preprocessing import StandardScaler

pd.set_option('display.max_columns', None) # Desactivamos el límite de columnas para que las muestre todas
pd.set_option('display.max_colwidth', None)

In [2]:
# 1. Cargamos el dataset original y mostramos las primeras 5 filas
df = pd.read_csv('games.csv')
df.head()

,AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies
2539430,Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0,0.00,0,0,NaN,[],[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/header.jpg?t=1699268702,NaN,NaN,NaN,True,False,False,0,NaN,0,0,0,NaN,0,0,NaN,0,0,0,0,NaN,NaN,NaN,NaN,NaN,"https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_7d9504b958d0b143d053d31cb74b375daba338d6.1920x1080.jpg?t=1699268702,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_85d2749d23e115b4d44e8ed1024c55a6be524fde.1920x1080.jpg?t=1699268702",NaN
496350,Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,0,5.24,65,0,"Springtime, April: when the cherry trees come into full bloom. The protagonist Yukinari Sanada has returned to his hometown in Kanagawa Prefecture, Kamakura City, for the first time in seven years, and is greeted by his older cousin Sakura Narumi (complete with maid outfit). He wanted to live in peace, but his life at the academy placed on the coastlands becomes bustling and brilliant while surrounded by a lineup of girls with booming personalities like Hotaru Amano, the sharp tongued, half-Japanese beauty, and Alice Kamishiro, the lazy witch who loves modern-science and mail-orders. Eventually, our protagonist is coaxed into joining the action committee for the academy's traditional beauty contest by his close friend. He interacts with the heroines participating in this contest who are cute but each have an idiocyncrasy or two. Fun events (with their own difficulties) pop up again and again. Your story with 'her' about smiles, peace, passion, and love, all while borrowing the power of a witch, is about to begin! 'Supipara - Alice the Magical Conductor.' is an interactive novel themed around 'dynamicism' depicted through a rallying all of the production techniques minori has accumulated to date. See expressive character portraits with moving eyes, mouths, even hair! It is quite immersive in many aspects. Also, 'Supipara' won't conclude in a single game. It will be released as several standalone episodes. Fans may freely pursue only the episodes that interest them if so desired. You could say 'it's a light novel series, and independent stories which develop their own unique personalities all in one.'",['English'],[],NaN,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/header.jpg?t=1725519097,http://mangagamer.org/supipara,http://mangagamer.com,support@mangagamer.com,True,False,False,0,NaN,0,252,3,NaN,0,231,NaN,8,0,8,0,minori,MangaGamer,"Single-player,Steam Trading Cards,Steam Cloud,Family Sharing",Adventure,"Adventure,Visual Novel,Anime,Cute","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_0a5ff85870a1c2221e495b75a5c7b9e248eb3249.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_f53e18a6c7be8eeb39e7c8efff8395b444786f68.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_1f86a8d171a549c504acd6592f559fc4bd9bd448.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_f40b6e8e535047b60412b8f102b907cb2317d53d.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_32c2149186b260804a2a8bed8e595f36228f8e16.1920x1080.jpg?t=1725519097,https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/496350/ss_abc2fe50c29b2621ae970f846a1e11d2e66c7394.1920x1080.jpg?t=1725

In [3]:
df.info()

<class 'pandas.DataFrame'>
Index: 125855 entries, 2539430 to 4247310
Data columns (total 39 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   AppID                       125854 non-null  str    
 1   Name                        125855 non-null  str    
 2   Release date                125855 non-null  str    
 3   Estimated owners            125855 non-null  int64  
 4   Peak CCU                    125855 non-null  int64  
 5   Required age                125855 non-null  float64
 6   Price                       125855 non-null  int64  
 7   DiscountDLC count           125855 non-null  int64  
 8   About the game              117393 non-null  str    
 9   Supported languages         125855 non-null  str    
 10  Full audio languages        125855 non-null  str    
 11  Reviews                     12181 non-null   str    
 12  Header image                125774 non-null  str    
 13  Website                

In [4]:
df.describe()

,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,Metacritic score,User score,Positive,Negative,Score rank,Achievements,Recommendations,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Movies
count,1.258550e+05,125855.000000,125855.000000,125855.000000,125855.000000,125855.000000,125855.000000,1.258550e+05,1.258550e+05,40.000000,125855.000000,1.258550e+05,1.258550e+05,125855.000000,1.258550e+05,125855.000000,0.0
mean,5.318720e+01,0.165365,4.811174,17.896826,0.535434,2.500083,0.023916,1.018182e+03,1.648786e+02,99.175000,17.922220,9.380399e+02,2.027568e+02,13.507711,1.691803e+02,14.419125,NaN
std,3.681083e+03,1.642666,12.489329,28.628518,14.328417,13.492922,1.376811,2.772782e+04,5.305001e+03,0.675107,139.696425,2.159585e+04,1.107222e+04,266.925759,1.105725e+04,290.747134,NaN
min,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,98.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
25%,0.000000e+00,0.000000,0.590000,0.000000,0.000000,0.000000,0.000000,0.000000e+00,0.000000e+00,99.000000,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
50%,0.000000e+00,0.000000,2.390000,0.000000,0.000000,0.000000,0.000000,4.000000e+00,1.000000e+00,99.000000,2.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
75%,0.000000e+00,0.000000,5.545000,37.000000,0.000000,0.000000,0.000000,3.500000e+01,1.000000e+01,100.000000,19.000000,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000,NaN
max,1.013936e+06,21.000000,999.980000,100.000000,3703.000000,97.000000,100.000000,7.642084e+06,1.173003e+06,100.000000,9821.000000,4.830455e+06,3.429544e+06,20088.000000,3.429544e+06,20088.000000,NaN


In [5]:
df.isnull().sum()

AppID                              1
Name                               0
Release date                       0
Estimated owners                   0
Peak CCU                           0
Required age                       0
Price                              0
DiscountDLC count                  0
About the game                  8462
Supported languages                0
Full audio languages               0
Reviews                       113674
Header image                      81
Website                        75335
Support url                    70560
Support email                  22630
Windows                            0
Mac                                0
Linux                              0
Metacritic score                   0
Metacritic url                121597
User score                         0
Positive                           0
Negative                           0
Score rank                    125815
Achievements                       0
Recommendations                    0
N

Sección: Preprocesamiento y Control de Calidad del Dataset
1. Diagnóstico de Anomalías en la Ingesta de Datos


Durante la fase inicial de carga del archivo games.csv mediante la librería pandas, se detectó un desalineamiento crítico en la matriz de datos. El síntoma principal fue el desplazamiento de los valores hacia la derecha a partir de la columna Price, lo que provocaba inconsistencias severas en el dominio de las variables (por ejemplo, fechas de lanzamiento indexadas como nombres de videojuegos, o registros numéricos incoherentes en la restricción de edad, como Required age = 64).

2. Análisis Técnico de la Causa Raíz

Para identificar el origen del desalineamiento, se realizó una inspección quirúrgica del archivo en texto plano (I/O puro de Python), aislando la estructura de la cabecera (header) respecto a las filas de registros físicos.

El análisis reveló un error de diseño estructural en el dataset de origen: el creador omitió una coma delimitadora en la cadena de la cabecera entre las variables Discount y DLC count, resultando en el string continuo DiscountDLC count.

Impacto técnico: La cabecera corrupta declaraba un total de 39 columnas, mientras que las filas de datos contenían en realidad 40 valores delimitados por comas. Al realizar el parsing estándar, el motor de Pandas sufría un desajuste posicional (data shifting), arrastrando cada celda una posición a la derecha a partir de la séptima columna.

3. Estrategia de Mitigación e Implementación

Para subsanar esta anomalía sin alterar la integridad del archivo original ni perder registros mediante técnicas drásticas de descarte, se diseñó un pipeline de ingesta personalizado:

Inyección de Cabecera Corregida: Se ignoró la línea de cabecera nativa del CSV mediante el parámetro header = 0 y se definió un array explícito de 40 columnas corregidas, restituyendo la separación formal entre Discount y DLC count.

Control de Comas Anidadas: Se forzó el uso del parámetro quotechar = '"' combinado con skipinitialspace = True para asegurar que las fechas con formato anglosajón (ej. "Aug 1, 2023") fueran interpretadas como un único bloque indivisible de texto y no como delimitadores de columna.

Filtrado Selectivo (Feature Selection Prerrequisito): Para optimizar la memoria volátil del entorno y evitar el ruido de strings masivos, se restringió la carga únicamente a las 12 variables críticas identificadas para el modelo de preventa.

In [6]:
# Mostramos las 2 primeras líneas reales del archivo en texto puro para analizar los datos y arreglarlos a mano
with open('games.csv', 'r', encoding = 'utf-8', errors = 'ignore') as f:
    cabecera = f.readline()
    primera_fila = f.readline()
print("--- CABECERA ORIGINAL DEL ARCHIVO ---")
print(cabecera)
print("\n--- PRIMERA FILA REAL DE DATOS ---")
print(primera_fila)

--- CABECERA ORIGINAL DEL ARCHIVO ---
AppID,Name,Release date,Estimated owners,Peak CCU,Required age,Price,DiscountDLC count,About the game,Supported languages,Full audio languages,Reviews,Header image,Website,Support url,Support email,Windows,Mac,Linux,Metacritic score,Metacritic url,User score,Positive,Negative,Score rank,Achievements,Recommendations,Notes,Average playtime forever,Average playtime two weeks,Median playtime forever,Median playtime two weeks,Developers,Publishers,Categories,Genres,Tags,Screenshots,Movies


--- PRIMERA FILA REAL DE DATOS ---
2539430,"Black Dragon Mage Playtest","Aug 1, 2023","0 - 0",0,0,0.0,0,0,"","[]","[]","","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/header.jpg?t=1699268702","","","",True,False,False,0,"",0,0,0,"",0,0,"",0,0,0,0,"","","","","","https://shared.akamai.steamstatic.com/store_item_assets/steam/apps/2539430/ss_7d9504b958d0b143d053d31cb74b375daba338d6.1920x1080.jpg?t=1699268702,https://shared.akamai.steamstat

In [7]:
# 2. Limpieza básica y asignación de índice. Definimos la lista de columnas corregida (reparando el error del creador del dataset)
columnas_corregidas = [
    'AppID', 'Name', 'Release date', 'Estimated owners', 'Peak CCU', 
    'Required age', 'Price', 'Discount', 'DLC count', 'About the game', 
    'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 
    'Website', 'Support url', 'Support email', 'Windows', 'Mac', 'Linux', 
    'Metacritic score', 'Metacritic url', 'User score', 'Positive', 'Negative', 
    'Score rank', 'Achievements', 'Recommendations', 'Notes', 
    'Average playtime forever', 'Average playtime two weeks', 
    'Median playtime forever', 'Median playtime two weeks', 
    'Developers', 'Publishers', 'Categories', 'Genres', 'Tags', 
    'Screenshots', 'Movies'
]

# Cargamos solo las columnas seguras que decidimos usar para la preventa
columnas_a_usar = [
    'Name', 'Release date', 'Estimated owners', 'Required age', 'Price', 
    'Windows', 'Mac', 'Linux', 'Developers', 'Publishers', 'Genres', 'Supported languages'
]

# Leemos el CSV saltándonos la cabecera rota (header = 0) y asignando nuestros nombres corregidos (names = columnas_corregidas)
df_bueno = pd.read_csv('games.csv', header = 0, names = columnas_corregidas, usecols = columnas_a_usar,
    quotechar = '"', skipinitialspace = True, on_bad_lines = 'skip'
)

# Limpiamos los datos y asignamos Name al índice
df_bueno.dropna(subset = ['Name'], inplace = True)
df_bueno.drop_duplicates(subset = ['Name'], inplace = True)
df_bueno.set_index('Name', inplace = True)
df_bueno.head()

,Release date,Estimated owners,Required age,Price,Supported languages,Windows,Mac,Linux,Developers,Publishers,Genres
Name,,,,,,,,,,,
Black Dragon Mage Playtest,"Aug 1, 2023",0 - 0,0,0.00,[],True,False,False,NaN,NaN,NaN
Supipara - Chapter 1 Spring Has Come!,"Jul 29, 2016",0 - 20000,0,5.24,['English'],True,False,False,minori,MangaGamer,Adventure
Mystery Solitaire The Black Raven,"May 6, 2019",0 - 20000,0,4.99,"['English', 'French', 'German', 'Russian']",True,True,False,Somer Games,8floor,Casual
버튜버 파라노이아 - Vtuber Paranoia,"Oct 31, 2024",0 - 20000,0,8.99,['Korean'],True,False,False,유진게임즈,유진게임즈,"Casual,Indie,Simulation"
Maze Quest VR,"Apr 24, 2025",0 - 20000,0,4.99,['English'],True,False,False,Reality Expanded LLC,Reality Expanded LLC,"Action,Early Access"


In [8]:
# 3. Creación del Target Multiclase. Función para limpiar el rango de texto y obtener el promedio numérico de owners
def transformar_owners_a_numerico(rango):
    if pd.isna(rango) or not isinstance(rango, str):
        return 0
    partes = rango.replace(',', '').split(' - ') # Limpiamos comas y separamos por el guion
    if len(partes) == 2:
        try:
            min_owners = float(partes[0])
            max_owners = float(partes[1])
            return (min_owners + max_owners) / 2
        except:
            return 0
    return 0

# Aplicamos la transformación para tener una métrica continua de ventas
df_bueno['owners_promedio'] = df_bueno['Estimated owners'].apply(transformar_owners_a_numerico)
condiciones = [ # Convertimos el valor numérico en las 3 clases de éxito
    (df_bueno['owners_promedio'] < 20000),
    (df_bueno['owners_promedio'] >= 20000) & (df_bueno['owners_promedio'] <= 200000),
    (df_bueno['owners_promedio'] > 200000)
]
df_bueno['exito_target'] = np.select(condiciones, [0, 1, 2], default = 0) # Asignamos las etiquetas (0, 1, 2)
print("Proporción de juegos por clase de éxito:")
print(df_bueno['exito_target'].value_counts(normalize = True) * 100)

Proporción de juegos por clase de éxito:
exito_target
0    79.674112
1    16.065755
2     4.260133
Name: proportion, dtype: float64


Sección: Definición y Formalización de la Variable Objetivo (Target)
1. Justificación Académica del Cambio de Target.

Inicialmente, el éxito de un videojuego se evaluaba mediante métricas cualitativas secundarias (como el ratio de reseñas positivas). Sin embargo, para dotar al proyecto de un enfoque comercial y de negocio riguroso, se redefinió la variable objetivo utilizando Estimated owners (Volumen estimado de propietarios) como el proxy definitivo de éxito de mercado.Predecir directamente las copias vendidas mediante una regresión lineal tradicional introduce un alto grado de variabilidad debido al comportamiento de los outliers (juegos virales o megaproducciones). Por lo tanto, se optó por un enfoque de Clasificación Multiclase, estratificando el mercado en tres segmentos competitivos bien diferenciados.

2. Criterios de Estratificación y Clases de Éxito.

Para evitar umbrales arbitrarios, se adoptaron los estándares de distribución de la industria y las recomendaciones del equipo docente, dividiendo el volumen continuo de propietarios en tres niveles indexados:
   Clase 0: Fracaso Comercial ($<20,000$ propietarios). Define el umbral crítico de la "larga cola" de Steam, donde se encuentra la gran mayoría de los proyectos independientes que no logran recuperar la inversión de desarrollo básica.
   Clase 1: Éxito Medio ($20,000 - 200,000$ propietarios). Representa la clase media de la plataforma; proyectos con una tracción comercial saludable, habitualmente asociados a estudios indies medianos o juegos de nicho de presupuesto moderado (AA)
   Clase 2: Gran Éxito ($>200,000$ propietarios). Identifica a los grandes dinamizadores del mercado: superproducciones AAA y fenómenos virales que logran un impacto masivo en ventas y sostenibilidad a largo plazo.

3. Tratamiento Matemático de Rangos.

Dado que la variable original se almacena como un rango categórico de texto (ej. "20000 - 50000"), el pipeline aplica una función de tipado que extrae los límites inferior y superior, calculando su punto medio aritmético como valor numérico continuo antes de realizar la segmentación por condiciones lógicas.

In [9]:
# 4. Feaute Engineering. Creación de variables. 
# Variables de Conteo Estructurado. Contamos cuántos idiomas y géneros tiene el juego (proxies de presupuesto y alcance)
# Usamos .fillna('') para convertir los valores nulos (NaN) en textos vacíos para evitar errores
# Aplicamos una función lambda que separa el texto por comas mediante .split(',') y averiguar el número de idiomas y géneros
df_bueno['num_idiomas'] = df_bueno['Supported languages'].fillna('').apply(lambda x: len(str(x).split(',')))
df_bueno['num_generos'] = df_bueno['Genres'].fillna('').apply(lambda x: len(str(x).split(',')))

# Convertimos las fechas para extraer mes y año. errors = 'coerce' pasa las fechas rotas a nulo sin romper el código
df_bueno['Release date'] = pd.to_datetime(df_bueno['Release date'], errors = 'coerce')
df_bueno['año_lanzamiento'] = df_bueno['Release date'].dt.year
df_bueno['mes_lanzamiento'] = df_bueno['Release date'].dt.month.fillna(1).astype(int)

# Variable derivada: Lanzamiento en temporada navideña/fiestas (Noviembre y Diciembre)
df_bueno['lanzamiento_navidad'] = df_bueno['mes_lanzamiento'].isin([11, 12]).astype(int)

# Mapeo de Experiencia y Tamaño del Estudio (Indie/AA/AAA). Contamos cuántos juegos ha publicado cada Publisher en todo el dataset histórico
dict_experiencia_pub = df_bueno['Publishers'].value_counts().to_dict()
df_bueno['juegos_publicados_historico'] = df_bueno['Publishers'].map(dict_experiencia_pub).fillna(1).astype(int)

# Clasificación del Estudio basada en su volumen de lanzamientos: 0 = Indie (1 a 9 juegos) | 1 = AA (10 a 50 juegos) | 2 = AAA (>50 juegos)
condiciones_estudio = [
    (df_bueno['juegos_publicados_historico'] > 50),
    (df_bueno['juegos_publicados_historico'] >= 10) & (df_bueno['juegos_publicados_historico'] <= 50),
    (df_bueno['juegos_publicados_historico'] < 10)
]
df_bueno['tipo_estudio'] = np.select(condiciones_estudio, [2, 1, 0], default = 0)

# Aseguramos el tipado numérico del Precio
df_bueno['Price'] = pd.to_numeric(df_bueno['Price'], errors = 'coerce').fillna(0.0)
df_bueno['Required age'] = pd.to_numeric(df_bueno['Required age'], errors = 'coerce').fillna(0).astype(int)
df_bueno.head()

,Release date,Estimated owners,Required age,Price,Supported languages,Windows,Mac,Linux,Developers,Publishers,Genres,owners_promedio,exito_target,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio
Name,,,,,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,2023-08-01,0 - 0,0,0.00,[],True,False,False,NaN,NaN,NaN,0.0,0,1,1,2023,8,0,1,0
Supipara - Chapter 1 Spring Has Come!,2016-07-29,0 - 20000,0,5.24,['English'],True,False,False,minori,MangaGamer,Adventure,10000.0,0,1,1,2016,7,0,77,2
Mystery Solitaire The Black Raven,2019-05-06,0 - 20000,0,4.99,"['English', 'French', 'German', 'Russian']",True,True,False,Somer Games,8floor,Casual,10000.0,0,4,1,2019,5,0,267,2
버튜버 파라노이아 - Vtuber Paranoia,2024-10-31,0 - 20000,0,8.99,['Korean'],True,False,False,유진게임즈,유진게임즈,"Casual,Indie,Simulation",10000.0,0,1,3,2024,10,0,13,1
Maze Quest VR,2025-04-24,0 - 20000,0,4.99,['English'],True,False,False,Reality Expanded LLC,Reality Expanded LLC,"Action,Early Access",10000.0,0,1,2,2025,4,0,1,0


Sanitización de Datos y Manejo de Valores Ausentes:
Para asegurar la compatibilidad con los algoritmos de Machine Learning, los valores nulos (NaN / NaT) fueron imputados bajo criterios lógicos de negocio: las anomalías en precios y restricciones de edad se fijaron en 0.0 (gratuito) y 0 (sin restricción) respectivamente; las inconsistencias en fechas se asignaron por defecto al mes 1 (enero); y los registros sin historial de distribuidor (Publisher) se inicializaron en 1, clasificándose automáticamente como desarrollos independientes (Indie).

Tratamiento y Normalización de Entidades (Developers):
En los datasets de plataformas de distribución digital (como Steam), el campo del desarrollador presenta una elevada cardinalidad y un alto nivel de ruido debido a discrepancias en el registro de nombres, fusiones de estudios, co-desarrollos y marcas regionales. Para garantizar que las variables de entrada aporten una señal limpia al modelo de aprendizaje automático, se diseñó e implementó un pipeline de procesamiento en tres fases:

1. Limpieza Textual y Eliminación de "Stop Words" de la Industria
En primera instancia, se aplicó una función de limpieza mediante expresiones regulares para estandarizar los nombres a formato Title Case y eliminar caracteres especiales.Asimismo, se identificó que términos genéricos de la industria como «Studios», «Games», «Interactive» o «Software» generaban falsas variaciones de una misma entidad. Tras extirpar estas palabras vacías (stop words específicas del dominio), se logró reducir significativamente el ruido contextual de los datos.

2. Detección de Duplicados mediante Similitud Difusa (Fuzzy Matching)
Para identificar variaciones ortográficas y pequeñas erratas de tipografía, se empleó un algoritmo de comparación difusa basado en la métrica de SequenceMatcher (difflib), utilizando un índice inverso por palabras clave para optimizar la complejidad computacional $O(N^2)$.
- Ajuste del Umbral: Al trabajar sobre texto previamente filtrado, se incrementó el umbral de similitud al 82% ($0.82$), evitando la inclusión de falsos positivos (como diferenciar estudios independientes con nombres similares).
- Escudo Numérico: Se integró una regla de bloqueo mediante expresiones regulares para evitar la unificación de entregas secuenciales o versiones numeradas.

3. Consolidación Semántica de Marcas y Colaboraciones
Dado que la comparación puramente textual no detecta relaciones corporativas ni co-desarrollos, se realizó una auditoría manual de los pares candidatos para aplicar una consolidación de marca:
- Consolidación de Marcas Principales: Múltiples variaciones regionales o de distribución (p. ej., Capcom Vancouver, Capcom USA, Capcom,Gogcom) se unificaron bajo la marca principal (Capcom).
- Atribución de Colaboraciones y Crossovers: En casos de grandes editoras o colectivos prolíficos (como Koei Tecmo, MicroProse, Cyanide o Sokpop Collective), las entradas compuestas por co-desarrollos se atribuyeron a la entidad principal para evitar la fragmentación de la autoría y la aparición de categorías testimoniales (long tail).

4. Codificación Numérica e Ingeniería de Características (Feature Engineering)
Los modelos de aprendizaje automático no pueden interpretar directamente cadenas de texto. Para transformar la variable categórica developer_clean en características numéricas ejecutables sin sufrir los problemas de alta dimensionalidad del One-Hot Encoding, se crearon dos nuevas variables predictoras:
- dev_volumen_juegos ($N$): Variable discreta que contabiliza el número histórico de títulos publicados por el desarrollador en la plataforma, representando la experiencia y presencia en el mercado.
- dev_exito_promedio: Variable continua en el rango $[0, 2]$ que mide la reputación y rendimiento histórico del desarrollador.

5. Suavizado Bayesiano (Bayesian Target Encoding)
Para evitar el sobreajuste (overfitting) en desarrolladores con muy pocos juegos (p. ej., un desarrollador con un único juego de éxito no debe recibir una puntuación perfecta de 2.0 por azar), se implementó un Target Encoding con Suavizado Bayesiano:
$$\text{Éxito Ajustado} = \frac{(N \times \text{Éxito Dev}) + (m \times \text{Éxito Global})}{N + m}$$
Donde $N$ es el volumen de juegos del desarrollador, $\text{Éxito Global}$ es la media del target en todo el dataset, y $m$ es el parámetro de suavizado (fijado en $m = 5$).
- Gestión de Desarrolladores Nuevos en Producción ($N = 0$): para garantizar la capacidad de generalización del modelo ante desarrolladores novatos o sin historial previo en la fase de inferencia, la fórmula asigna automáticamente la media global de la industria (dev_exito_promedio = $\text{Media Global}$) y establece dev_volumen_juegos = $0$. De este modo, se evita penalizar arbitrariamente con un cero a un creador emergente, tratándolo como una incertidumbre neutral ponderada por el mercado.

In [10]:
# 5. Limpieza completa y normalización de los datos para lectura del modelo. Usamos 'Capcom' para demostrar que hay duplicidades en el dataset
# Filtramos las filas donde la columna 'developer' contenga ese texto (sin importar mayúsculas/minúsculas)
resultados = df_bueno[df_bueno['Developers'].str.contains("Capcom", case = False, na = False)]
print("Nombres exactos encontrados en el dataset:")
print(resultados['Developers'].unique())

Nombres exactos encontrados en el dataset:
<StringArray>
[                  'Capcom Vancouver',                             'Capcom',
                   'CAPCOM Co., Ltd.',             'Capcom,Digital Eclipse',
                    'CAPCOM CO., LTD',                'Capcom U.S.A., Inc.',
 'Capcom Game Studio Vancouver, Inc.',                             'CAPCOM',
                   'CAPCOM CO., LTD.',       'Capcom Game Studio Vancouver',
          'Double Helix Games,Capcom',           'CAPCOM Co., Ltd.,GOG.com']
Length: 12, dtype: str


In [11]:
def limpiar_nombre_developer(nombre): # Si la celda está vacía, es nula o no es texto, devolvemos NaN
    if not isinstance(nombre, str) or nombre.strip() == "":
        return np.nan
    partes = nombre.split(',') # Si hay múltiples developers separados por coma, los limpiamos por separado para no mezclar developers
    partes_limpias = []
    for parte in partes:
        p = parte.strip()
        p = unicodedata.normalize('NFKC', p) # Mantenemos caracteres no latinos
        p = p.replace('®', '').replace('™', '').replace('©', '') # Eliminar marcas registradas y símbolos comunes
        p = ''.join(c for c in p if unicodedata.category(c)[0] in ('L', 'N') or c.isspace() or c == "-") # Conserva letras, números, espacios, guiones
        patron_limpieza = r'\b(co|ltd|inc|gmbh|llc|sa|corp|corporation|holdings|sl|srl|pvt|association|group|)\b' # Quitar siglas y palabras comunes
        p = re.sub(patron_limpieza, '', p, flags = re.IGNORECASE)
        p = ' '.join(p.split()) # Quitar espacios dobles que queden tras la limpieza
        if p: # Si tras la limpieza queda algo de texto
            partes_limpias.append(p.title()) # Formato "Título" (Capcom, Ubisoft)
    if not partes_limpias: # Si después de limpiar el texto se queda vacío (ej: la celda era solo "Co., Ltd.")
        return np.nan
    return ','.join(partes_limpias) # Unimos con una coma limpia sin espacios (estándar del Excel)

df_bueno['developer_clean'] = df_bueno['Developers'].apply(limpiar_nombre_developer)
df_bueno['publisher_clean'] = df_bueno['Publishers'].apply(limpiar_nombre_developer)
print(df_bueno[df_bueno['Developers'].str.contains('KOEI TECMO', case = False, na = False)]['developer_clean'].unique())
print(df_bueno[df_bueno['Developers'].str.contains('Capcom', case = False, na = False)]['developer_clean'].unique())

<StringArray>
[                         'Koei Tecmo Games',
              'Square Enix,Koei Tecmo Games',
                                'Koei Tecmo',
         'Koei Tecmo Games,Line Games,Motif',
 'Platinumgames Team Ninja Koei Tecmo Games',
               'Team Ninja,Koei Tecmo Games']
Length: 6, dtype: str
<StringArray>
[            'Capcom Vancouver',                       'Capcom',
       'Capcom,Digital Eclipse',                   'Capcom Usa',
 'Capcom Game Studio Vancouver',    'Double Helix Games,Capcom',
                'Capcom,Gogcom']
Length: 7, dtype: str


In [12]:
# Buscamos posibles duplicados usando una comparación difusa (fuzzy matching). Agrupamos nombres que se parezcan un 82% o más
def similar(a, b):
    return SequenceMatcher(None, a, b).ratio()

counts = df_bueno['developer_clean'].value_counts()
todos_los_devs = [dev for dev in counts.index.tolist() if isinstance(dev, str)]
duplicados_sospechosos = set()
indice_palabras = defaultdict(list)
for dev in todos_los_devs:
    palabras = set(re.split(r'[\s,]+', dev.lower()))
    for palabra in palabras:
        if len(palabra) > 2: # Evitamos letras sueltas o palabras de 2 letras
            indice_palabras[palabra].append(dev)

for palabra, num_devs in indice_palabras.items():
    if len(num_devs) < 2: # Si solo hay un desarrollador con esa palabra, no hay duplicado
        continue
    if len(num_devs) > 40: # Evitamos palabras ultra-comunes que queden en el dataset
        continue
    for i in range(len(num_devs)):
        for j in range(i + 1, len(num_devs)):
            dev1 = num_devs[i]
            dev2 = num_devs[j]
            pareja_id = tuple(sorted([dev1, dev2]))
            if pareja_id in duplicados_sospechosos:
                continue
            nums1 = set(re.findall(r'\d+', dev1)) # Si tienen números diferentes (ej: 'Version 5' vs 'Version 6'), no se unifican
            nums2 = set(re.findall(r'\d+', dev2))
            if nums1 and nums2 and nums1 != nums2:
                continue  # Salta a la siguiente pareja directamente
            ratio = similar(dev1, dev2)
            if ratio >= 0.82: 
                duplicados_sospechosos.add(pareja_id)

resultado_final = [(d1, d2, similar(d1, d2)) for d1, d2 in duplicados_sospechosos]
resultado_final.sort(key = lambda x: x[2], reverse = True)
for dev1, dev2, score in resultado_final:
    print(f"¿Unificar? -> '{dev1}' con '{dev2}' (Similitud: {score:.2f})")

¿Unificar? -> 'Trend Readaktions- Und Verlagsgesellschaft Mbh' con 'Trend Redaktions- Und Verlagsgesellschaft Mbh' (Similitud: 0.99)
¿Unificar? -> 'Two Thousand And One Entertainment Studio' con 'Two Thousand And One Entertainment Studios' (Similitud: 0.99)
¿Unificar? -> 'Italian Party Of Indie Developer' con 'Italian Party Of Indie Developers' (Similitud: 0.98)
¿Unificar? -> 'Metthink,Matheus Reis Queiroga' con 'Metthink,Mattheus Reis Queiroga' (Similitud: 0.98)
¿Unificar? -> 'Random Thoughts Enterainment' con 'Random Thoughts Entertainment' (Similitud: 0.98)
¿Unificar? -> 'Just Add Water Development' con 'Just Add Water Developments' (Similitud: 0.98)
¿Unificar? -> 'Hong Haryong,Jung Youngju' con 'Hong Haryong,Jung Younju' (Similitud: 0.98)
¿Unificar? -> 'Interactive Wave Studio' con 'Interactive Wave Studios' (Similitud: 0.98)
¿Unificar? -> 'Laush Dmitriy Segeevich' con 'Laush Dmitriy Sergeevich' (Similitud: 0.98)
¿Unificar? -> 'Worldgame Entertainement' con 'Worldgame Entertainment

In [13]:
# Buscamos filas donde distintos developers colaboradores coincidan para reducir el tamaño de los developers únicos
top_devs = df_bueno['developer_clean'].dropna().unique()
top_devs = [d for d in top_devs if isinstance(d, str) and len(d) > 3]
num_tandas = 6
tamano_tanda = len(top_devs) // num_tandas
tandas_disponibles = {
    f"tanda_{i + 1}": top_devs[i * tamano_tanda : (i + 1) * tamano_tanda if i < num_tandas - 1 else len(top_devs)]
    for i in range(num_tandas)
}
limite_visual = 25
contador_resultados = 0
nombre_tanda_actual = "tanda_1" # De tanda_1 a tanda_6, 11666 developers por tanda
semillas_actuales = tandas_disponibles[nombre_tanda_actual]
print(f"🚀 Ejecutando {nombre_tanda_actual} ({len(semillas_actuales)} semillas a evaluar)...\n")
devs_serie = df_bueno['developer_clean'].dropna()
resultados_parciales = []
for dev_principal in semillas_actuales:
    patron = rf'\b{re.escape(dev_principal)}\b'
    mascara = devs_serie.str.contains(patron, case = False, na = False)
    coincidencias = devs_serie[mascara].unique()
    if len(coincidencias) > 1:
        print(f"🏢 Estudio Principal: '{dev_principal}'")
        print(f"    Variantes encontradas ({len(coincidencias)}): {list(coincidencias)}")
        print("-" * 50)
        contador_resultados += 1 # Rompemos el bucle en cuanto alcanzamos los 25 impresos para no colapsar Jupyter
        if contador_resultados >= limite_visual:
            print(f"✨ Se han mostrado los primeros {limite_visual} resultados.")
            break

🚀 Ejecutando tanda_1 (11666 semillas a evaluar)...

🏢 Estudio Principal: 'Airem'
    Variantes encontradas (6): ['Airem', 'Àirȩm,Airem', 'Retroairem,Airem', 'Impossible,Airem', 'Cats,Airem', 'Airem Motorsport Tech,Airem']
--------------------------------------------------
🏢 Estudio Principal: 'Owned By Gravity'
    Variantes encontradas (2): ['Owned By Gravity', 'Owned By Gravity,Ulisses Digital']
--------------------------------------------------
🏢 Estudio Principal: 'Zeno'
    Variantes encontradas (4): ['Zeno', 'Zeno Rogue', 'Thomas Biskup,Jochen Terstiege,Zeno Rogue,Krzysztof Dycha,Lucas Dieguez', 'Zeno-Daniel Görbe']
--------------------------------------------------
🏢 Estudio Principal: 'Accolade'
    Variantes encontradas (5): ['Accolade', 'Compro Games,Accolade', 'Distinctive Software,Accolade,Big Boat Interactive', 'Artech Digital Entertainments,Accolade', 'Cyberlore Studios,Accolade']
--------------------------------------------------
🏢 Estudio Principal: 'Jolly Crouton Media

In [14]:
# Habiendo analizado todos los cambios, unificamos una gran cantidad de developers. "Original", "Nuevo dev"
unificaciones_completas_lista = [
    ("Àirȩm,Airem", "Airem"),
    ("Retroairem,Airem", "Airem"),
    ("Airem Motorsport Tech,Airem", "Airem"),
    ("Owned By Gravity,Ulisses Digital", "Owned By Gravity"),
    ("Compro Games,Accolade", "Accolade"),
    ("Artech Digital Entertainments,Accolade", "Accolade"),
    ("Cyberlore Studios,Accolade", "Accolade"),
    ("Jolly Crouton Media,Tiny Warrior Games", "Jolly Crouton Media"),
    ("Square Enix,Gemdrops", "Square Enix"),
    ("Tecopark,Gemdrops", "Gemdrops"),
    ("Deeber Studio", "Deeber"),
    ("Taito,M2", "Taito"),
    ("The Gentlebros,General Arcade", "The Gentlebros"),
    ("Anomalybyte Studio,Sunset Disco", "Anomalybyte Studio"),
    ("Hgmgame,Hgmgame", "Hgmgame"),
    ("Hgmgame,Bird Gogogo", "Hgmgame"),
    ("Starbreeze Studios Ab", "Starbreeze Studios"),
    ("O3 Games,Starbreeze Studios,Zoom Platform Media,Jordan Freeman", "Starbreeze Studios"),
    ("Chequered Ink,Gamedevdan", "Chequered Ink"),
    ("Sleepwalkstudio,橙光游戏", "橙光游戏"),
    ("上海酷影,橙光游戏,Sleepwalkstudio", "橙光游戏"),
    ("Prismic Studios,Dano Kablamo", "Prismic Studios"),
    ("Weathered Sweater,Aerialknight,Titan Arx Interactive", "Weathered Sweater"),
    ("Space Boat Studios,Baked Games", "Space Boat Studios"),
    ("Dreamstudios,Sandstorm In A Bottle", "Sandstorm In A Bottle"),
    ("Outlier", "Outlier Interactive"),
    ("Red Android,Epixr Games Ug", "Epixr Games Ug"),
    ("Crowz Games,Fhnbhj", "Fhnbhj"),
    ("Scs Software,Sunstorm Interactive", "Scs Software"),
    ("Mossmouth,Blitworks", "Mossmouth"),
    ("Infinity Ward,Raven Software,Beenox,Treyarch,High Moon Studios,Sledgehammer Games,Demonware,Toys For Bob", "Raven Software"),
    ("Sledgehammer Games,Treyarch,Infinity Ward,Beenox,Raven Software,High Moon Studios,Demonware", "Raven Software"),
    ("Sledgehammer Games,Treyarch,High Moon Studios,Raven Software,Beenox,Demonware", "Raven Software"),
    ("Sledgehammer Games,Raven Software", "Raven Software"),
    ("Treyarch,Raven Software,Beenox,High Moon Studios,Sledgehammer Games,Infinity Ward,Activision Shanghai,Demonware", "Activision"),
    ("Infinity Ward,Beenox,High Moon Studios,Raven Software", "Raven Software"),
    ("Treyarch,Raven Software,Beenox,High Moon Studios,Activision Shanghai,Sledgehammer Games", "Activision"),
    ("Treyarch,Raven Software,Beenox,High Moon Studios,Activision Shanghai,Sledgehammer Games,Infinity Ward,Demonware", "Activision"),
    ("Raven Software,Beenox", "Raven Software"),
    ("Infinity Ward,Raven Software,Beenox,Treyarch,High Moon Studios,Sledgehammer Games,Activision Shanghai,Demonware,Toys For Bob", "Activision"),
    ("I M Game,I M Fine", "I M Game"),
    ("Alith Games,Artless Games", "Artless Games"),
    ("Daruma The Factory", "Daruma"),
    ("Soup With Friends,Eggnut", "Eggnut"),
    ("Boom Games Team", "Boom Games"),
    ("Pyramid,G-Mode", "G-Mode"),
    ("G-Mode,Atlus", "G-Mode"),
    ("G-Mode,Marvelous", "Marvelous"),
    ("G-Mode,Room6", "G-Mode"),
    ("G-Mode,Fromsoftware", "Fromsoftware"),
    ("Room6,G-Mode", "G-Mode"),
    ("G-Mode,Hopemoon", "G-Mode"),
    ("Force Of Habit,Clockwork Cuckoo", "Force Of Habit"),
    ("Infogrames Europe,Etranges Libellules Studios", "Etranges Libellules Studios"),
    ("For Fun Labs", "Fun Labs"),
    ("Nanningsgames,Sascha Van Huis", "Nanningsgames"),
    ("Nanningsgames,Fromhomegames", "Nanningsgames"),
    ("Jiffy Goat,The Garden", "Jiffy Goat"),
    ("Redkar Devs,Unknown Studios", "Redkar Devs"),
    ("Independent Arts Software", "Independent Arts"),
    ("Bonus Level Entertainment,Independent Arts Software", "Independent Arts"),
    ("Independent Arts Software,Upjers", "Independent Arts"),
    ("Jerzie,Creobit", "Creobit"),
    ("Workroom7,Creobit", "Creobit"),
    ("Toyman,Creobit", "Creobit"),
    ("Good Games,Creobit", "Creobit"),
    ("Creobit,Workroom7", "Creobit"),
    ("Ejaw,Creobit", "Creobit"),
    ("Unibit,Creobit", "Creobit"),
    ("Creobit,Gameon Production", "Creobit"),
    ("Lightup Promotion", "Lightup"),
    ("Fire Totem Games", "Totem Games"),
    ("The Dreamers Guild,Qubyte Interactive", "Qubyte Interactive"),
    ("Qubyte Interactive,Mito Games", "Qubyte Interactive"),
    ("Flip Axis Studios", "Axis Studios"),
    ("Haemimont Games,Abstraction", "Haemimont Games"),
    ("Madmind Studio After Hours", "Madmind Studio"),
    ("Fissure,Gonyagjelly,Legendsetter,Lornapan", "Fissure"),
    ("Vitok,Kedexa", "Kedexa"),
    ("Mancebo Games,Rendercode Games", "Rendercode Games"),
    ("Static City Games,Indiemyke", "Static City Games"),
    ("Static City Games,Vampiric Games", "Static City Games"),
    ("Initial F Studio", "F Studio"),
    ("Amazing Seasun Games", "Seasun Games"),
    ("Seasun Games Pte", "Seasun Games"),
    ("Battle Factory,Hexwar Games", "Hexwar Games"),
    ("Bookmark Games,Hexwar Games", "Hexwar Games"),
    ("Balaber1Yl,B52 Development Team", "B52 Development Team"),
    ("Webfoot Games", "Webfoot"),
    ("Studio2760,Dnovel", "Dnovel"),
    ("Dnovel,Freeanimalssoftware", "Dnovel"),
    ("Dnovel,Dnovel", "Dnovel"),
    ("Imposant Entertainment,Dnovel", "Dnovel"),
    ("Freeanimalssoftware,Dnovel", "Dnovel"),
    ("Imlbe,Dnovel", "Dnovel"),
    ("Dnovel,Two Hands", "Dnovel"),
    ("Acid Arrow Studio,Sons Of Welder,Fm Simple Games Studio", "Sons Of Welder,Fm Simple Games Studio"),
    ("The House Of Fables,Gogii Games", "Gogii Games"),
    ("Gogii Games,World-Loom", "Gogii Games"),
    ("Powerhoof,Dave Lloyd", "Powerhoof"),
    ("Ssi,Strategic Simulations", "Strategic Simulations"),
    ("Strategic Simulations,Micromagic", "Strategic Simulations"),
    ("Strategic Simulations,Doug Wood", "Strategic Simulations"),
    ("Westwood Associates,Strategic Simulations", "Westwood Associates"),
    ("Stormfront Studios,Strategic Simulations", "Strategic Simulations"),
    ("Mcon,Lost Rabbit Digital", "Lost Rabbit Digital"),
    ("Boomzap Entertainment", "Boomzap"),
    ("Boomzap,Gamehouse", "Boomzap"),
    ("迩来工作室,Windfield Games", "Windfield Games"),
    ("Evgeny Osmet,Ez Games", "Ez Games"),
    ("343 Industries,Splash Damage,Ruffian Games,Bungie,Saber Interactive", "Saber Interactive"),
    ("Crytek,Saber Interactive", "Saber Interactive"),
    ("Saber Interactive,Redemption Ark", "Saber Interactive"),
    ("Flaming Fowl Studios,Saber Interactive", "Saber Interactive"),
    ("343 Industries,Saber Interactive,Bungie", "Saber Interactive"),
    ("Omnis Rector Studios", "Omnis"),
    ("Finji,Kittehface Software", "Kittehface Software"),
    ("Play2Chill,Atomic Jelly", "Atomic Jelly"),
    ("Team Fusion Simulations,1C Maddox Games", "1C Maddox Games"),
    ("Kyuu Fujisaki,Minimol Games", "Minimol Games"),
    ("Wayforward,Limited Run Games", "Wayforward"),
    ("Abstraction Games,Wayforward", "Wayforward"),
    ("13Am Games,Wayforward", "Wayforward"),
    ("Wayforward,Abstraction Games", "Wayforward"),
    ("Eidos-Montréal,Crystal Dynamics,Nixxes,Feral Interactive Mac,Feral Interactive Linux", "Eidos-Montréal"),
    ("Crystal Dynamics,Eidos-Montréal,Feral Interactive Mac,Nixxes", "Eidos-Montréal"),
    ("Obsidian Entertainment,Eidos-Montréal", "Eidos-Montréal"),
    ("Eidos-Montréal,Feral Interactive Mac", "Eidos-Montréal"),
    ("Crystal Dynamics,Eidos-Montréal,Feral Interactive Mac,Feral Interactive Linux,Nixxes", "Eidos-Montréal"),
    ("Chibig,Nukefist", "Chibig"),
    ("Chibig,Talpa Games,Undercoders", "Chibig"),
    ("Crazylabs,Vectormonk", "Crazylabs"),
    ("Clockstone Software", "Clockstone"),
    ("Capcom Vancouver", "Capcom"),
    ("Capcom,Digital Eclipse", "Capcom"),
    ("Capcom Usa", "Capcom"),
    ("Capcom Game Studio Vancouver", "Capcom"),
    ("Double Helix Games,Capcom", "Capcom"),
    ("Capcom,Gogcom", "Capcom"),
    ("Mission Critical Studios,More Hay", "Mission Critical Studios"),
    ("Mission Critical Studios,Handcrafted Mystical Games,Normal Distribution", "Mission Critical Studios"),
    ("Gaterooze,Ink,Microprose Software", "Microprose Software"),
    ("Microprose Software,Digital Theory", "Microprose Software"),
    ("Deadly Games,Microprose Software", "Microprose Software"),
    ("Realtime Games,Microprose Software", "Microprose Software"),
    ("Why485,Microprose Software", "Microprose Software"),
    ("World-Loom,Gamehouse", "World-Loom"),
    ("World-Loom,Nitreal", "World-Loom"),
    ("World-Loom,The House Of Fables", "World-Loom"),
    ("Flightless Squid Studios", "Flightless"),
    ("Klabater,Juggler Games", "Klabater"),
    ("Cabbibo,Mikael Emtinger", "Cabbibo"),
    ("Sarah Hashkes,Matthew Hoe,Isaac Cohen Cabbibo,Atley Loughridg", "Cabbibo"),
    ("Square Enix,Koei Tecmo Games", "Koei Tecmo Games"),
    ("Koei Tecmo Games,Line Games,Motif", "Koei Tecmo Games"),
    ("Platinumgames Team Ninja Koei Tecmo Games", "Koei Tecmo Games"),
    ("Team Ninja,Koei Tecmo Games", "Koei Tecmo Games"),
    ("Sir-Tech,Madlab Software", "Sir-Tech"),
    ("Sir-Tech Canada", "Sir-Tech"),
    ("Polaristm Team,Variable State", "Variable State"),
    ("Ubisoft Quebec,Ubisoft Montreal,Ubisoft Bucharest,Ubisoft Singapore,Ubisoft Montpellier,Ubisoft Kiev,Ubisoft Shanghai", "Ubisoft"),
    ("Ubisoft Quebec,Ubisoft Belgrade,Ubisoft Bordeaux,Ubisoft Bucharest Craiova,Ubisoft Chengdu,Ubisoft Montpellier,Ubisoft Montreal,Ubisoft Osaka,Ubisoft Philippines,Ubisoft Shanghai,Ubisoft Singapore,Ubisoft Sofia,Ubisoft Ukraine", "Ubisoft"),
    ("Merlyn", "Merlyn Labs"),
    ("Eipix Entertainment,Blue Tea Games", "Eipix Entertainment"),
    ("Blue Tea Games,Eipix Entertainment", "Eipix Entertainment"),
    ("Live Wire,Adglobe", "Live Wire"),
    ("Live Wire,Treasure", "Live Wire"),
    ("Adglobe,Live Wire", "Live Wire"),
    ("アトリエさくら Atelier Sakura", "Atelier Sakura"),
    ("Devs United Games", "United Games"),
    ("The Software Farm,Sunlight Games,Sierra Entertainment", "Sunlight Games"),
    ("Pillow Fight,Worst Girls Games", "Pillow Fight"),
    ("Smallbü,Pillow Fight", "Pillow Fight"),
    ("Cosen,Talesshop", "Talesshop"),
    ("Archor Wright,Hoguru Games", "Archor Wright"),
    ("Archor Wright,Zenn Games", "Archor Wright"),
    ("Red Bull Media House,Red Bull,Koffeecup", "Koffeecup"),
    ("Playgendary,Joybits", "Joybits"),
    ("Xinfinity Games,Nova Studios", "Xinfinity Games"),
    ("Cateia Games,Gamehouse", "Cateia Games"),
    ("Cateia Games,Autumn Moon Entertainment,Jordan Freeman", "Cateia Games"),
    ("Three Headed Eagle,Cateia Games,Goldfinch Studios", "Cateia Games"),
    ("United Independent Entertainment,Crafty Studios", "United Independent Entertainment"),
    ("Introversion Software,Double Eleven", "Introversion Software"),
    ("Rsu Horizon,Good Job Multimedia,Pixel Perfex", "Pixel Perfex"),
    ("Square Enix,Human Head Studios", "Square Enix"),
    ("Inxile Entertainment,Krome Studios", "Inxile Entertainment"),
    ("Krome Studios,Inxile Entertainment", "Inxile Entertainment"),
    ("Sokpop Collective,Tom Van Den Boogaart", "Sokpop Collective"),
    ("Adriaan De Jongh,Sokpop Collective", "Sokpop Collective"),
    ("Sokpop Collective,Sokpop Collective", "Sokpop Collective"),
    ("Quantic Dream,Aspyr Media Remastered Version", "Quantic Dream"),
    ("Quantic Dream Sas", "Quantic Dream"),
    ("Daedalic Entertainment,Irresponsible Games", "Daedalic Entertainment"),
    ("Koboldgames,Daedalic Entertainment", "Daedalic Entertainment"),
    ("Teyon,Competition Company", "Teyon"),
    ("Tate Multimedia,Teyon", "Teyon"),
    ("Sleepteam,Zima Software", "Zima Software"),
    ("Honeyslug,Richard Hogg", "Honeyslug"),
    ("Good Day Games,Nyx Digital", "Nyx Digital"),
    ("Ghost Clusters,Nyx Digital", "Nyx Digital"),
    ("Inlogic Games,Nyx Digital", "Nyx Digital"),
    ("Handsome Box,Cinemax Games", "Cinemax Games"),
    ("Rainbow Rhino,Cinemax Games", "Cinemax Games"),
    ("Hyperbolic Magnetism,Cinemax Games", "Cinemax Games"),
    ("Wales Interactive,Good Gate Media", "Wales Interactive"),
    ("Dark Rift Horror,Wales Interactive", "Wales Interactive"),
    ("Splendy Interactive,Wales Interactive", "Wales Interactive"),
    ("Wales Interactive,Good Gate Media,Little Jade Productions", "Wales Interactive"),
    ("Wales Interactive,Dead Pixel Productions,Good Gate Media", "Wales Interactive"),
    ("Wales Interactive,Interflix Media", "Wales Interactive"),
    ("Davekki Studios,Wales Interactive", "Wales Interactive"),
    ("Wales Interactive,Good Gate Media,Wayout Pictures,Posterity Entertainment", "Wales Interactive"),
    ("Phantom Studio,Not The Moon Studio", "Phantom Studio"),
    ("Deep Cave Studios", "Deep Cave"),
    ("Andreev Worlds,Gamesforgames", "Gamesforgames"),
    ("Space R Studio,Gamesforgames", "Gamesforgames"),
    ("Maks Volegov,Gamesforgames", "Gamesforgames"),
    ("Succubella Games,Gamesforgames", "Gamesforgames"),
    ("Scale Studio,Gamesforgames", "Gamesforgames"),
    ("Korion Interactive,Nementic Games", "Korion Interactive"),
    ("Nementic Games,Korion Interactive", "Korion Interactive"),
    ("Manodeix,Hymoori", "Hymoori"),
    ("Games Incubator,Frozen Cave Studio", "Games Incubator"),
    ("Games Incubator,Gameformatic", "Games Incubator"),
    ("Games Incubator,North Wind", "Games Incubator"),
    ("Games Incubator,Hypnotic Ants", "Games Incubator"),
    ("Monuments Games,Games Incubator", "Games Incubator"),
    ("Kawaiinium,Lump Of Sugar", "Lump Of Sugar"),
    ("Azarashi Software,Lump Of Sugar", "Lump Of Sugar"),
    ("Lump Of Sugar,Hikari Field", "Lump Of Sugar"),
    ("The Naked Dev,Alterego Games", "Alterego Games"),
    ("9Ratones,Leaf Team Games", "Leaf Team Games"),
    ("喵呜喵5,0Cube", "0Cube"),
    ("Grouch,Suitntie,Negative Karma", "Grouch,Suitntie"),
    ("Cyanide Studio,Umix Studios", "Cyanide Studio"),
    ("Umix Studios,Cyanide Studio", "Cyanide Studio"),
    ("Leikir Studio,Cyanide Studio", "Cyanide Studio"),
    ("Cyanide Studio,Spiders", "Cyanide Studio"),
    ("Black Shamrock,Cyanide Studio", "Cyanide Studio"),
    ("Red Mountain Game Design", "Red Mountain"),
    ("Red Mountain Games,Ritual Interactive", "Red Mountain"),
    ("Qumaron,Truegames", "Qumaron"),
    ("Gotcha Gotcha Games,Kadokawa,Yoji Ojima", "Kadokawa"),
    ("Gotcha Gotcha Games,フォトショ職人,Kadokawa", "Kadokawa"),
    ("City Connection,Kadokawa", "Kadokawa"),
    ("Kadokawa,Tsukurite", "Kadokawa"),
    ("Gotcha Gotcha Games,Kadokawa", "Kadokawa"),
    ("Gotcha Gotcha Games,ゆわか,Kadokawa", "Kadokawa"),
    ("Gotcha Gotcha Games,エイジエジェネスタ,Kadokawa", "Kadokawa"),
    ("Gotcha Gotcha Games,道楽,Kadokawa", "Kadokawa"),
    ("Kadokawa,Active Gaming Media", "Kadokawa"),
    ("Game Studio,Kadokawa", "Kadokawa"),
    ("Kadokawa,Gotcha Gotcha Games", "Kadokawa"),
    ("Kadokawa Games Grasshopper Manufacture", "Grasshopper Manufacture"),
    ("Gotcha Gotcha Games,湿度ケイ,Kadokawa", "Kadokawa"),
    ("Nayug,Kadokawa", "Kadokawa"),
    ("Digital Melody,Storm Trident S A", "Digital Melody"),
    ("Wastelands Interactive,Digital Melody", "Digital Melody"),
    ("Digital Melody,Wastelands Interactive", "Digital Melody"),
    ("Pestoforce,Skeleton Crew Studios,Poxpower", "Pestoforce"),
    ("Cosen,Talesshop", "Cosen"),
    ("Softwaves,New Games Waves", "Softwaves"),
    ("Softwaves Mini", "Softwaves"),
    ("New Games Waves,Softwaves", "Softwaves"),
    ("Gg Interactive,Gg Studio", "Gg Interactive"),
    ("Yustas Game Studio,Alawar Entertainment", "Alawar Entertainment"),
    ("Wellore,Alawar Entertainment", "Alawar Entertainment"),
    ("Alawar Stargaze,Alawar Entertainment", "Alawar Entertainment"),
    ("Adept Studios Gd,Alawar Entertainment", "Alawar Entertainment"),
    ("Alawar Entertainment,Amegami", "Alawar Entertainment"),
    ("New Bridge Games,Alawar Entertainment", "Alawar Entertainment"),
    ("Nova Ideals,Alawar Entertainment", "Alawar Entertainment"),
    ("Boolat Games,Alawar Entertainment", "Alawar Entertainment"),
    ("Rionix,Alawar Entertainment", "Alawar Entertainment"),
    ("Toyman Interactive,Alawar Entertainment", "Alawar Entertainment"),
    ("Alawar Entertainment,Toyman Interactive", "Alawar Entertainment"),
    ("Ikinagames,Nanali Studios", "Nanali Studios"),
    ("Immersive Vr Education Plc", "Immersive Vr Education"),
    ("Jim Dex Games", "Jim Dex"),
    ("Artcelerator,Pantyparrot", "Pantyparrot"),
    ("Gamechanger Studio,Neon Doctrine", "Gamechanger Studio"),
    ("Gamechanger Studio,Toge Productions", "Gamechanger Studio"),
    ("Gentledriver,Puzzle L Porject,河野", "Gentledriver,Puzzle L Porject"),
    ("Soma Games,Clopas", "Soma Games"),
    ("Sparkypants Studios", "Sparkypants"),
    ("Tml-Studios,Pedepe", "Tml-Studios"),
    ("Shoker Games,Burrik", "Burrik"),
    ("Wicked Games Workshop", "Wicked Games"),
    ("Housemarque,Climax Studios", "Housemarque"),
    ("Ratz N Godz,Storybird Games", "Storybird"),
    ("Storybird Games", "Storybird"),
    ("Storybird Studio", "Storybird"),
    ("Storybird,Julien Rocca", "Storybird"),
    ("Spaceboy Games", "Spaceboy"),
    ("Play,Silden", "Silden"),
    ("Monstars,Resonair", "Resonair"),
    ("Monstars,Resonair,Stage Games", "Resonair"),
    ("French-Bread,Arc System Works", "Arc System Works"),
    ("Arc System Works,Toybox", "Arc System Works"),
    ("Cygames,Arc System Works", "Arc System Works"),
    ("Arc System Works,Dotemu", "Arc System Works"),
    ("Circle Ent,Arc System Works,Initory Studio", "Arc System Works"),
    ("Fk Digital,Arc System Works", "Arc System Works"),
    ("Arc System Works,Aplus", "Arc System Works"),
    ("Neople,Arc System Works,Eighting", "Arc System Works"),
    ("Arc System Works,Atlus", "Arc System Works"),
    ("Grove Street Games,E Mcneill", "E Mcneill"),
    ("Mi-Clos Studio,Goblinz Studio,Fibretigre", "Goblinz Studio"),
    ("Dark Root Gardeners,Goblinz Studio", "Goblinz Studio"),
    ("Nihon Falcom,Mastiff", "Nihon Falcom"),
    ("Nihon Falcom,Ph3", "Nihon Falcom"),
    ("Nihon Falcom,Engine Software Bv,Ph3", "Nihon Falcom"),
    ("Nihon Falcom,Ph3,Engine Software Bv", "Nihon Falcom"),
    ("Tom Create,株式会社トムクリエイト", "Tom Create"),
    ("Lgt Sia,Httpswwwfacebookcomlgtsia", "Lgt Sia"),
    ("Gameon Production Kz", "Gameon Production"),
    ("Creobit,Gameon Production", "Creobit"),
    ("Thiemo Bolder,Onemangames", "Onemangames"),
    ("Atmos Games,Serenity Forge", "Atmos Games"),
    ("Notgames,Babaroga", "Babaroga"),
    ("Bloom Technology,Blueside", "Blueside"),
    ("3Ichael 7Ambert,A Rockets Intensity", "3Ichael 7Ambert"),
    ("3Ichael 7Ambert,Rameez,Matt Mateo Vii,Rayu Johnson", "3Ichael 7Ambert"),
    ("久川はる,園児ニア", "園児ニア"),
    ("Sports Interactive,Creative Assembly,Rovio Sweden Ab,Torn Banner,Team 17,Sumo Digital,Bossa,Curve Digital,Hardlight,Spilt Milk Studios,Modern Dream", "Sports Interactive"),
    ("Mysterytag,Argali Entertainment", "Mysterytag"),
    ("Far Mills,Mysterytag", "Mysterytag"),
    ("Mysterytag,Far Mills", "Mysterytag"),
    ("Fabsquare,Orbi Universo Team", "Orbi Universo Team"),
    ("Brainstorming Team,Kuko", "Brainstorming Team"),
    ("Ironcode Gaming", "Ironcode"),
    ("Orangutan Matter", "Orangutan"),
    ("Kink Master Studios,Takeover Studios", "Kink Master Studios"),
    ("Ds-Sans,Ebi-Hime", "Ebi-Hime"),
    ("Sinking Sheep,Dionous Games", "Sinking Sheep"),
    ("Swimming Scorpions,Mmeu", "Mmeu"),
    ("Minsan Pte", "Minsan"),
    ("Magnolia Games,Enhydra Games,Chocolate Ship Games", "Chocolate Ship Games"),
    ("Josyan,Bibiki", "Josyan"),
    ("Josyan,Bibiki,Karv,Gabriel Machado", "Josyan"),
    ("Gibar,Reanimatorus", "Gibar"),
    ("雍乐蜜獾,Happy Honey Badger", "Honey Badger"),
    ("Avelar Studios,Vaulted Secrets", "Avelar Studios"),
    ("Vrai-Dev", "Vrai"),
    ("Mindware Studios", "Mindware"),
    ("Frontwing,枕", "Frontwing"),
    ("Blank-Note Frontwing", "Frontwing"),
    ("Old School Vibes,Gray Boss", "Gray Boss"),
    ("Sonic Team,Bitbaboon", "Sonic Team"),
    ("Sonic Team,Hardlight", "Sonic Team"),
    ("Square Enix,Artdink", "Square Enix"),
    ("Epyx,Retro Room Games", "Epyx"),
    ("Blind Mind Studios,Glacicle", "Blind Mind Studios"),
    ("Lilith Guild", "Lilith"),
    ("Vivi,Dana", "Vivi"),
    ("Spoonfed Interactive,Combat Waffle Studios", "Combat Waffle Studios"),
    ("1C Game Studios,777 Studios", "1C Game Studios"),
    ("Colloseusx,Cpmilan", "Colloseusx"),
    ("Torus Games,Pyro Studios", "Pyro Studios"),
    ("Yippee Entertainment,Pyro Studios", "Pyro Studios"),
    ("Arkhouse Telegraph", "Arkhouse"),
    ("Charlykuschel,Pocketgames", "Charlykuschel"),
    ("Locomalito,Gryzor87,Abylight", "Locomalito,Gryzor87"),
    ("Sky Hour Works,Anarch Entertainment", "Sky Hour Works"),
    ("Pidroh,Perennial Hearts", "Perennial Hearts"),
    ("Old School Vibes,Motion Game Studio", "Old School Vibes"),
    ("G-Mode,Marvelous", "Marvelous"),
    ("Marvelous Games", "Marvelous"),
    ("Marvelous,Brownies", "Marvelous"),
    ("Snowhaven Studios,Crystal Game Works", "Snowhaven Studios"),
    ("Hash Technology,哈视奇科技", "哈视奇科技"),
    ("Bandai Namco Studios,Bandai Namco Online", "Bandai Namco Studios"),
    ("Bandai Namco Studios Vancouver,Hipster Whale,3 Sprockets", "Bandai Namco Studios"),
    ("Qloc,Bandai Namco Studios", "Bandai Namco Studios"),
    ("Bandai Namco Studios Singapore Pte,Bandai Namco Studios Malaysia Sdn Bhd", "Bandai Namco Studios"),
    ("Bandai Namco Studios,Dimps", "Bandai Namco Studios"),
    ("Jiangli,Gooodmon", "Jiangli"),
    ("Farmind Studio,100 Games", "Farmind Studio"),
    ("Loopy Games,Cortexa233,Koway,Psalms77,Hooke", "Koway"),
    ("Grasshopper Manufacture,Active Gaming Media", "Grasshopper Manufacture"),
    ("Grasshopper Manufacture,Playism", "Grasshopper Manufacture"),
    ("Grasshopper Manufacture,Engine Software Bv", "Grasshopper Manufacture"),
    ("Digital Reality,Grasshopper Manufacture,Gyroscope Games", "Grasshopper Manufacture"),
    ("Grasshopper Manufacture,Supertrick Games", "Grasshopper Manufacture"),
    ("Mydreamforever,Tazdraperm", "Mydreamforever"),
    ("Freeworld Developments,Burning Scale Studios", "Freeworld Developments"),
    ("Funselektor Labs,Flippfly", "Flippfly"),
    ("Nightshade Interactive", "Nightshade"),
    ("Mgames Studio", "Mgames"),
    ("Ghxyk2,Bankbank", "Bankbank"),
    ("Louisselle,Minicactus Games", "Minicactus Games"),
    ("Zgame Studio", "Zgamestudio"),
    ("Ubisoft Toronto", "Ubisoft"),
    ("Ubisoft Sofia", "Ubisoft"),
    ("Ubisoft Singapore", "Ubisoft"),
    ("Ubisoft Shanghaï", "Ubisoft"),
    ("Ubisoft San Francisco", "Ubisoft"),
    ("Ubisoft Romania", "Ubisoft"),
    ("Ubisoft Quebec", "Ubisoft"),
    ("Ubisoft Paris Studio", "Ubisoft"),
    ("Ubisoft Paris", "Ubisoft"),
    ("Ubisoft Montreal Studio", "Ubisoft"),
    ("Ubisoft Montréal", "Ubisoft"),
    ("Ubisoft Montreal", "Ubisoft"),
    ("Ubisoft Montpellier", "Ubisoft"),
    ("Ubisoft Mainz", "Ubisoft"),
    ("Ubisoft Kiev", "Ubisoft"),
    ("Ubisoft Ivory Tower", "Ubisoft"),
    ("Ubisoft Entertainment", "Ubisoft"),
    ("Ubisoft Düsseldorf", "Ubisoft"),
    ("Ubisoft Bordeaux", "Ubisoft"),
    ("Ubisoft Blue Byte", "Ubisoft"),
    ("Ubisoft Annecy", "Ubisoft"),
    ("Ubisoft Abu Dhabi", "Ubisoft"),
    ("Ubisoft - San Francisco", "Ubisoft"),
    ("Trend Readaktions- Und Verlagsgesellschaft Mbh", "Trend Redaktions- Und Verlagsgesellschaft Mbh"),
    ("Sega Studios Australia", "Sega"),
    ("Sega Hardlighttm", "Sega"),
    ("Punkcake Delicieux", "Punkcake Délicieux"),
    ("Popcap", "Popcap Games"),
    ("Microprose", "Microprose Software"),
    ("Microids Studio Paris", "Microids"),
    ("Microids Studio Lyon", "Microids"),
    ("Megapixel Studio S A", "Megapixel"),
    ("Megapixel Studio", "Megapixel"),
    ("Massive Entertainment A Ubisoft Studio", "Massive Entertainment"),
    ("Marginalact", "Marginal Act"),
    ("Male Doll Stories", "Male Doll"),
    ("Male Doll Adventures", "Male Doll"),
    ("Laush Dmitriy Sergeevich", "Laush Studio"),
    ("Konami Digital Entertainment", "Konami"),
    ("Koei Tecmo", "Koei Tecmo Games"),
    ("Feral Interactive Mac", "Feral Interactive"),
    ("Feral Interactive Linux", "Feral Interactive"),
    ("Epixr Games Ug", "Epixr Games"),
    ("Elephant Games Ar", "Elephant Games"),
    ("Elephant Ar", "Elephant Games"),
    ("Elephant", "Elephant Games"),
    ("Electronic Arts - Tiburon", "Electronic Arts"),
    ("Elaphant Ar", "Elephant Games"),
    ("Elaphant", "Elephant Games"),
    ("Efunsoft", "Efunsoft Games"),
    ("Ea Redwood Shores", "Electronic Arts"),
    ("Ea Phenomic", "Electronic Arts"),
    ("Ea Pacific", "Electronic Arts"),
    ("Ea Los Angeles", "Electronic Arts"),
    ("Ea Canada Ea Romania", "Electronic Arts"),
    ("Ea Canada", "Electronic Arts"),
    ("Ea - Maxis", "Electronic Arts"),
    ("E-Funsoft Games", "Efunsoft Games"),
    ("E-Funsoft", "Efunsoft Games"),
    ("Daedalic Studio West", "Daedalic Entertainment"),
    ("Cyanide Studio", "Cyanide Studios"),
    ("Cyanide Montreal", "Cyanide Studios"),
    ("Cyanide", "Cyanide Studios"),
    ("Croteam Vr", "Croteam"),
    ("Bandai Namco Forge Digitals", "Bandai Namco Studios"),
    ("Bandai Namco Entertainment", "Bandai Namco Studios"),
    ("Bandai Namco Studio", "Bandai Namco Studios"),
    ("Bandai Namco Amusement Lab", "Bandai Namco Studios"),
    ("Alawar Stargaze Warm Lamp Games", "Alawar Entertainment"),
    ("Alawar Stargaze", "Alawar Entertainment"),
    ("Alawar Southpoint", "Alawar Entertainment"),
    ("Alawar North", "Alawar Entertainment"),
    ("Alawar Fridays Games", "Alawar Entertainment"),
    ("Alawar Dreamdale", "Alawar Entertainment"),
    ("Alawar Casual", "Alawar Entertainment"),
    ("Alawar", "Alawar Entertainment"),
    ("Rockstar Studios", "Rockstar Games"),
    ("Rockstar North Toronto", "Rockstar Games"),
    ("Rockstar North", "Rockstar Games"),
    ("Rockstar New England", "Rockstar Games"),
    ("Bethesda Softworks", "Bethesda"),
    ("", ""),
    ("", "")
]

mapeo_unificaciones = dict(unificaciones_completas_lista) # Convertimos la lista de tuplas en un diccionario de mapeo
df_bueno['developer_clean'] = df_bueno['developer_clean'].replace(mapeo_unificaciones)
df_bueno['publisher_clean'] = df_bueno['publisher_clean'].replace(mapeo_unificaciones)
total_devs_unicos = df_bueno['developer_clean'].nunique() # Número final de developers unificados
print(f"Número de developers original: {df['Developers'].nunique()}")
print(f"Tras la limpieza masiva, quedan {total_devs_unicos} desarrolladores distintos en el dataset.")

Número de developers original: 72859
Tras la limpieza masiva, quedan 70736 desarrolladores distintos en el dataset.


In [15]:
def get_most_frequent_dev(cell): # Función para developers (selección por frecuencia)
    if pd.isna(cell) or not cell:
        return cell
    devs = str(cell).split(',')
    return max(devs, key = lambda d: dev_counts.get(d, 0))

def get_best_publisher(row): # Función para publishers (prioridad match dev > frecuencia)
    pub_cell = row['publisher_clean']
    dev_cell = row['developer_clean']
    if pd.isna(pub_cell) or not pub_cell:
        return pub_cell
    pubs = str(pub_cell).split(',') # Prioridad absoluta si el editor coincide con el desarrollador del juego
    if pd.notna(dev_cell):
        devs = str(dev_cell).split(',')
        for p in pubs:
            if p in devs:
                return p # Si no coincide ninguno, selecciona el publisher más frecuente del dataset
    return max(pubs, key = lambda p: pub_counts.get(p, 0))

# Normalización de las nuevas variables developer_clean y publisher_clean para la lectura del modelo, y limpieza de valores nulos. Suavizado Bayesiano
media_global_exito = df_bueno['exito_target'].mean()
m = 5 # Peso, 5 juegos para alejarse de la media global

# Fórmula Bayesiana para developers
stats_devs = df_bueno.groupby('developer_clean').agg(exito_dev = ('exito_target', 'mean'), n_juegos = ('exito_target', 'count'))
stats_devs['dev_exito_bayesiano'] = ((stats_devs['n_juegos'] * stats_devs['exito_dev']) + (m * media_global_exito))/(stats_devs['n_juegos'] + m)
df_bueno['dev_exito_promedio'] = df_bueno['developer_clean'].map(stats_devs['dev_exito_bayesiano'])
df_bueno['dev_volumen_juegos'] = df_bueno['developer_clean'].map(stats_devs['n_juegos'])

# Suavizado Bayesiano para publishers
stats_pubs = df_bueno.groupby('publisher_clean').agg(exito_pub = ('exito_target', 'mean'), n_juegos = ('exito_target', 'count'))
stats_pubs['pub_exito_bayesiano'] = ((stats_pubs['n_juegos'] * stats_pubs['exito_pub']) + (m * media_global_exito))/(stats_pubs['n_juegos'] + m)
df_bueno['pub_exito_promedio'] = df_bueno['publisher_clean'].map(stats_pubs['pub_exito_bayesiano'])
df_bueno['pub_volumen_juegos'] = df_bueno['publisher_clean'].map(stats_pubs['n_juegos'])

# Limpieza de valores nulos
df_bueno['dev_exito_promedio'] = df_bueno['dev_exito_promedio'].fillna(media_global_exito)
df_bueno['dev_volumen_juegos'] = df_bueno['dev_volumen_juegos'].fillna(0)
df_bueno['pub_exito_promedio'] = df_bueno['pub_exito_promedio'].fillna(media_global_exito)
df_bueno['pub_volumen_juegos'] = df_bueno['pub_volumen_juegos'].fillna(0)

# Contamos las frecuencias globales de cada entidad individual
all_devs = [dev for sublist in df_bueno['developer_clean'].dropna().str.split(',') for dev in sublist]
dev_counts = Counter(all_devs)
all_pubs = [pub for sublist in df_bueno['publisher_clean'].dropna().str.split(',') for pub in sublist]
pub_counts = Counter(all_pubs)
df_bueno['developer_clean'] = df_bueno['developer_clean'].apply(get_most_frequent_dev)
df_bueno['publisher_clean'] = df_bueno.apply(get_best_publisher, axis = 1)
print(df_bueno[df_bueno['developer_clean'].str.contains('KOEI TECMO', case = False, na = False)]['developer_clean'].unique())
print(df_bueno[df_bueno['developer_clean'].str.contains('Capcom', case = False, na = False)]['developer_clean'].unique())

<StringArray>
['Koei Tecmo Games']
Length: 1, dtype: str
<StringArray>
['Capcom']
Length: 1, dtype: str


In [17]:
# Lista de columnas a eliminar definitivamente
columnas_a_borrar = [
    'Release date', # Ya extraímos año y mes
    'Estimated owners', # Ya creamos 'exito_target' a partir de ella
    'owners_promedio', # Columna numérica intermedia, no la queremos para entrenar
    'Supported languages', 'Genres', # Ya tenemos 'num_idiomas' y 'num_generos'
    'Publishers', 'Developers', # Ya tenemos 'tipo_estudio' y el conteo histórico
    'developer_clean', 'publisher_clean' # Hizimos la limpieza masiva y lo pasamos a las variables éxito promedio y volumen juegos
]

# Aplicamos el drop sobre el DataFrame y comprobamos las columnas definitivas con las que se entrenará el modelo
df_modelo = df_bueno.drop(columns = columnas_a_borrar)
print("Columnas finales para el modelo:")
print(df_modelo.columns.tolist())

Columnas finales para el modelo:
['Required age', 'Price', 'Windows', 'Mac', 'Linux', 'exito_target', 'num_idiomas', 'num_generos', 'año_lanzamiento', 'mes_lanzamiento', 'lanzamiento_navidad', 'juegos_publicados_historico', 'tipo_estudio', 'dev_exito_promedio', 'dev_volumen_juegos', 'pub_exito_promedio', 'pub_volumen_juegos']


In [18]:
cols_os = ['Windows', 'Mac', 'Linux']
df_modelo[cols_os] = df_modelo[cols_os].astype(int) # Convertimos directamente True/False a 1/0
df_modelo.head()

,Required age,Price,Windows,Mac,Linux,exito_target,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
Name,,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,0,0.00,1,0,0,0,1,1,2023,8,0,1,0,0.245860,0.0,0.245860,0.0
Supipara - Chapter 1 Spring Has Come!,0,5.24,1,0,0,0,1,1,2016,7,0,77,2,0.171485,8.0,0.210113,77.0
Mystery Solitaire The Black Raven,0,4.99,1,1,0,0,4,1,2019,5,0,267,2,0.056391,70.0,0.097798,345.0
버튜버 파라노이아 - Vtuber Paranoia,0,8.99,1,0,0,0,1,3,2024,10,0,13,1,0.068295,13.0,0.068295,13.0
Maze Quest VR,0,4.99,1,0,0,0,1,2,2025,4,0,1,0,0.175614,2.0,0.204884,1.0


Reducción de Dimensionalidad y Eliminación de Redundancias:
Una vez finalizada la ingeniería de variables, se procedió a depurar el DataFrame eliminando las columnas cualitativas originales de texto plano (Genres, Publishers, Supported languages) y los campos intermedios de negocio (Release date, Estimated owners). Este paso es crítico para consolidar una matriz de características puramente numérica, optimizar el uso de memoria en el entorno de desarrollo y mitigar el riesgo de data leakage durante el entrenamiento del modelo.

In [19]:
df_modelo.info()

<class 'pandas.DataFrame'>
Index: 124644 entries, Black Dragon Mage Playtest to Project Liminal 2
Data columns (total 17 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Required age                 124644 non-null  int64  
 1   Price                        124644 non-null  float64
 2   Windows                      124644 non-null  int64  
 3   Mac                          124644 non-null  int64  
 4   Linux                        124644 non-null  int64  
 5   exito_target                 124644 non-null  int64  
 6   num_idiomas                  124644 non-null  int64  
 7   num_generos                  124644 non-null  int64  
 8   año_lanzamiento              124644 non-null  int32  
 9   mes_lanzamiento              124644 non-null  int64  
 10  lanzamiento_navidad          124644 non-null  int64  
 11  juegos_publicados_historico  124644 non-null  int64  
 12  tipo_estudio                 124644 no

In [20]:
# Separamos las características (X) y el objetivo (y) antes de normalizar
X = df_modelo.drop(columns = ['exito_target'])
y = df_modelo['exito_target']

# Estandarizamos las variables numéricas continuas (Media = 0, Varianza = 1). Solo escalamos las que no son booleanas/binarias
columnas_a_escalar = [
    'Price', 'Required age', 'num_idiomas', 'num_generos', 'juegos_publicados_historico', 'año_lanzamiento',
    'dev_exito_promedio', 'dev_volumen_juegos', 'pub_exito_promedio', 'pub_volumen_juegos'
]
scaler = StandardScaler()
X[columnas_a_escalar] = scaler.fit_transform(X[columnas_a_escalar])
print(f"Dataset listo. Características (X): {X.shape}, Target (y): {y.shape}")
X.head()

Dataset listo. Características (X): (124644, 16), Target (y): (124644,)


,Required age,Price,Windows,Mac,Linux,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
Name,,,,,,,,,,,,,,,,
Black Dragon Mage Playtest,-0.098826,-0.384657,1,0,0,-0.318697,-1.267932,0.407933,8,0,-0.314067,0,-0.086947,-0.350864,-0.145072,-0.333254
Supipara - Chapter 1 Spring Has Come!,-0.098826,0.033542,1,0,0,-0.318697,-1.267932,-1.714291,7,0,1.202642,2,-0.510511,0.010135,-0.308648,1.146850
Mystery Solitaire The Black Raven,-0.098826,0.013590,1,1,0,-0.096888,-1.267932,-0.804767,5,0,4.994414,2,-1.165963,2.807878,-0.822600,6.298380
버튜버 파라노이아 - Vtuber Paranoia,-0.098826,0.332827,1,0,0,-0.318697,0.175330,0.711108,10,0,-0.074587,1,-1.098171,0.235759,-0.957607,-0.083366
Maze Quest VR,-0.098826,0.013590,1,0,0,-0.318697,-0.546301,1.014282,4,0,-0.314067,0,-0.486992,-0.260615,-0.332580,-0.314032


Tratamiento de Valores Críticos y Estandarización de Escalas:

Como último filtro de robustez, se interceptaron posibles valores infinitos (inf / -inf) transformándolos a nulos y aplicando una política de tolerancia cero mediante la imputación final a 0.
Para mitigar el impacto de las diferencias de magnitud entre las variables (por ejemplo, el rango de precios frente a las variables booleanas de plataformas), se aplicó un escalado empleando un StandardScaler. Este procedimiento transforma las distribuciones de las variables continuas para que presenten una media igual a $0$ y una varianza unitaria ($1$), garantizando la estabilidad numérica y un cálculo óptimo de las distancias en los algoritmos basados en gradientes y árboles.

Validación de la Política de Imputación frente a la Media Aritmética:

Se evaluó la viabilidad de utilizar la media estadística para la sustitución de valores ausentes, descartándose en favor de una imputación lógica por variables. Imputar la media en características como Price o Required age alteraría artificialmente la naturaleza de los datos (transformando juegos Free-to-Play en títulos de precio promedio, o introduciendo restricciones de edad inexistentes). Por lo tanto, se optó por una estrategia de valores mínimos absolutos (0 y 1), preservando la semántica del negocio de los videojuegos y evitando el sesgo en la distribución de las variables predictoras.

Interpretación del Espacio de Características Estandarizado:

La presencia de valores negativos en la matriz de características estandarizada ($X$) no representa una anomalía en los datos de origen (como precios o edades negativas), sino que es el resultado intrínseco de la transformación mediante StandardScaler. Al desplazar el centro de la distribución a una media aritmética de $\mu = 0$, el signo del valor estandarizado actúa como un indicador posicional: los valores negativos representan registros que se sitúan cuantitativamente por debajo de la media global del mercado de Steam, mientras que los valores positivos denotan registros que superan dicho promedio.

In [21]:
X.describe()

,Required age,Price,Windows,Mac,Linux,num_idiomas,num_generos,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,juegos_publicados_historico,tipo_estudio,dev_exito_promedio,dev_volumen_juegos,pub_exito_promedio,pub_volumen_juegos
count,1.246440e+05,1.246440e+05,124644.000000,124644.000000,124644.000000,1.246440e+05,1.246440e+05,1.246440e+05,124644.000000,124644.000000,1.246440e+05,124644.000000,1.246440e+05,1.246440e+05,1.246440e+05,1.246440e+05
mean,5.974205e-17,-2.918695e-17,0.999647,0.172475,0.127515,-5.974205e-17,7.547564e-17,5.910358e-15,6.671954,0.176206,3.921997e-17,0.308077,-2.225505e-16,5.016508e-18,-2.061329e-16,-6.384646e-18
std,1.000004e+00,1.000004e+00,0.018785,0.377794,0.333551,1.000004e+00,1.000004e+00,1.000004e+00,3.462484,0.380997,1.000004e+00,0.628285,1.000004e+00,1.000004e+00,1.000004e+00,1.000004e+00
min,-9.882576e-02,-3.846571e-01,0.000000,0.000000,0.000000,-3.186967e-01,-1.267932e+00,-7.474613e+00,1.000000,0.000000,-3.140674e-01,0.000000,-1.457565e+00,-3.508644e-01,-1.246385e+00,-3.332539e-01
25%,-9.882576e-02,-3.375697e-01,1.000000,0.000000,0.000000,-3.186967e-01,-5.463008e-01,-5.015919e-01,4.000000,0.000000,-3.140674e-01,0.000000,-4.869920e-01,-3.057395e-01,-4.665142e-01,-3.140317e-01
50%,-9.882576e-02,-1.939134e-01,1.000000,0.000000,0.000000,-3.186967e-01,1.753304e-01,4.079327e-01,7.000000,0.000000,-2.941107e-01,0.000000,-3.203067e-01,-3.057395e-01,-3.325798e-01,-2.948096e-01
75%,-9.882576e-02,6.147565e-02,1.000000,0.000000,0.000000,-9.688767e-02,8.969615e-01,7.111076e-01,10.000000,0.000000,-1.943272e-01,0.000000,9.986025e-02,-1.703649e-01,1.152593e-01,-1.986990e-01
max,1.292957e+01,7.942282e+01,1.000000,1.000000,1.000000,7.222811e+00,1.172143e+01,1.317457e+00,12.000000,1.000000,1.072199e+01,2.000000,7.321460e+00,1.011811e+01,6.148736e+00,1.031580e+01


In [24]:
# 6. Entrenamiento del modelo y prueba de algoritmos

# Módulo de Extracción y Preparación de Datos Futuros (Steam Web API)

## 1. Descripción General y Arquitectura
Para evaluar y predecir el comportamiento de lanzamientos futuros en la plataforma Steam, se diseñó un pipeline de extracción automatizado estructurado en **dos fases secuenciales**:

1. **Fase de Recolección de AppIDs (Fase 1):** Captura masiva de identificadores únicos de proyectos en desarrollo (*Popular Coming Soon*).
2. **Fase de Extracción de Características y Filtrado (Fase 2):** Consulta detallada a la API de Steam (`appdetails`), limpieza de datos, parseo defensivo de fechas y filtrado de contenido no deseado.

---

## 2. Fase 1: Recolección Masiva de AppIDs
A través del endpoint de búsqueda de la tienda de Steam (`/search/results/`), se extraen los identificadores primarios (`AppID`) de los juegos próximos más populares.

* **Paginación Dinámica:** Peticiones paginadas en bloques de 50 elementos por llamada (`count=50`), iterando el parámetro `start`.
* **Proporción de Recolección (5.5x):** Debido a que el catálogo no procesado contiene un alto volumen de expansiones (DLCs), demostraciones y títulos con información incompleta, se estableció un multiplicador de seguridad de 5,5. Para un objetivo final de 1.000 juegos limpios, se recolectan aproximadamente **5.500 AppIDs únicos**.
* **Gestión de Sesión:** Incorporación de cabeceras de navegador (`User-Agent`) y pausas de 1,2 segundos por página para prevenir bloqueos por tasa de peticiones (*Rate Limiting*).

---

## 3. Fase 2: Extracción de Características y Reglas de Filtrado
Cada `AppID` recolectado se procesa de forma individual mediante el endpoint público `https://store.steampowered.com/api/appdetails`.

### A. Estrategia de Filtrado y Limpieza
1. **Tipo de Producto:** Filtrado estricto (`data.get('type') == 'game'`) para descartar DLCs, software, bandas sonoras o demostraciones.
2. **Control de Errores y HTTP 429:** Implementación de manejo de excepciones para códigos de respuesta `429 (Too Many Requests)`. Si el servidor solicita enfriamiento, el hilo se pausa 10 segundos sin interrumpir la ejecución ni perder el lote.
3. **Pausas de Ejecución (0,35 s):** Latencia idónea por consulta que maximiza el rendimiento (~1 hora de ejecución total) manteniendo la estabilidad del scraping.

### B. Tratamiento Defensivo de Fechas y Parseo Combinado (Regex + Fallback)
El procesamiento de la fecha de lanzamiento (`release_date`) representó uno de los principales retos técnicos debido a la disparidad de formatos que utilizan los desarrolladores en Steam (*"Q4 2026"*, *"Coming Soon"*, *"2026"*, *"TBA"*).

* **Parseo Estándar:** Intento inicial de conversión mediante `pd.to_datetime`.
* **Procesamiento de Cadenas Indeterminadas:** Si el parseo estándar falla, se ejecuta un análisis mediante expresiones regulares (`re.findall(r'20\d{2}', raw_date)`):
  * Si detecta un año de lanzamiento explícito $\le 2026$ (sin mes exacto parseable) o años anómalos ($> 2030$), la fecha se reasigna automáticamente a **2027**.
  * Si la cadena carece de año (`"Coming Soon"`), se asigna el año **2027** por defecto.
* **Filtro de Descarte Inminente:** Títulos con fecha confirmada anterior o igual a julio de 2026 y años anteriores (`(year == 2026 and month <= 7) or (year < 2026)`) son descartados para asegurar que el dataset contenga exclusivamente lanzamientos futuros.

### C. Variables Extraídas para el Modelo
Por cada juego válido, se construye una estructura de datos estructurada con las siguientes variables:

* **Generales:** `AppID`, `Name`, `Release date`.
* **Temporales:** `año_lanzamiento`, `mes_lanzamiento`, `lanzamiento_navidad` (indicador binario para los meses 11 y 12).
* **Comerciales y Plataforma:** `Price` (convertido de céntimos a divisa estándar), compatibilidad con `Windows`, `Mac` y `Linux`.
* **Metadatos Agregados:** `Developers`, `Publishers`, `num_generos` (conteo de categorías) y `num_idiomas` (parseo y limpieza de etiquetas HTML de idiomas soportados).

---

## 4. Resumen de Métricas del Pipeline

| Métrica | Valor Estimado / Real |
| :--- | :--- |
| **AppIDs recolectados (Fase 1)** | ~5.500 identificadores |
| **Páginas analizadas (Fase 1)** | ~120 páginas |
| **Tasa de descarte (Morralla / Filtros)** | ~80% - 82% |
| **Dataset final resultante** | **~1.000 juegos base limpios** |
| **Tiempo total de ejecución** | ~55 - 60 minutos |

In [43]:
# 7. Creación del dataset de predicciones futuras. Petición a la API del buscador de Steam
headers = { # Simulamos un navegador real para evitar bloqueos
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Accept-Language': 'es-ES,es;q=0.9,en;q=0.8'
}
app_ids = []
target_games = 1000 # Cantidad objetivo de juegos, no recolectar más
current_page = 1
print("Obteniendo lista de AppIDs...") # Queremos ~1000 juegos limpios -> Recolectamos ~5500 AppIDs (multiplicador x5.5)
while len(app_ids) < target_games * 5.5 and current_page <= 120: # Endpoint de búsqueda con filtro de 'popular_coming_soon'
    search_url = f"https://store.steampowered.com/search/results/?query&start={(current_page - 1) * 50}&count=50&dynamic_data=&snr=1_7_7_230_13&filter=popularcomingsoon&infinite=1"
    try:
        response = requests.get(search_url, headers = headers, timeout = 10)
        if response.status_code == 200:
            res = response.json()
            html_content = res.get('results_html', '')
            found_ids = re.findall(r'data-ds-appid="(\d+)"', html_content) # Extraer todos los data-ds-appid del HTML que devuelve la tienda
            for app_id in found_ids: # Algunas respuestas incluyen múltiples IDs juntas (ej. "123,456"), tomamos la primera
                clean_id = int(app_id.split(',')[0])
                if clean_id not in app_ids:
                    app_ids.append(clean_id)
            print(f"Página {current_page}: {len(app_ids)} AppIDs acumulados.")
            current_page += 1
            time.sleep(1.2) # Pausa para no saturar la API (Rate Limit)
        else:
            print(f"Página {current_page}: Status HTTP {response.status_code}. Reintentando...")
            time.sleep(3) # Pausa por juego para no saturar la API
    except Exception as e:
        print(f"Error en la página {current_page}: {e}")
        time.sleep(3) # Pausa por juego para no saturar la API

print(f"Se han recolectado {len(app_ids)} AppIDs únicos.") # De todos estos, se descartarán el ~80% en el siguiente bloque (de ~5500 a ~1000)

Obteniendo lista de AppIDs...
Página 1: 50 AppIDs acumulados.
Página 2: 100 AppIDs acumulados.
Página 3: 150 AppIDs acumulados.
Página 4: 200 AppIDs acumulados.
Página 5: 250 AppIDs acumulados.
Página 6: 300 AppIDs acumulados.
Página 7: 350 AppIDs acumulados.
Página 8: 400 AppIDs acumulados.
Página 9: 450 AppIDs acumulados.
Página 10: 500 AppIDs acumulados.
Página 11: 550 AppIDs acumulados.
Página 12: 600 AppIDs acumulados.
Página 13: 650 AppIDs acumulados.
Página 14: 700 AppIDs acumulados.
Página 15: 750 AppIDs acumulados.
Página 16: 800 AppIDs acumulados.
Página 17: 850 AppIDs acumulados.
Página 18: 900 AppIDs acumulados.
Página 19: 950 AppIDs acumulados.
Página 20: 998 AppIDs acumulados.
Página 21: 1048 AppIDs acumulados.
Página 22: 1098 AppIDs acumulados.
Página 23: 1148 AppIDs acumulados.
Página 24: 1198 AppIDs acumulados.
Página 25: 1248 AppIDs acumulados.
Página 26: 1298 AppIDs acumulados.
Página 27: 1348 AppIDs acumulados.
Página 28: 1398 AppIDs acumulados.
Página 29: 1448 AppI

In [45]:
games_list = []
print("Extrayendo y procesando características desde Steam...")
for idx, app_id in enumerate(app_ids):
    if len(games_list) >= target_games:
        break # Alcanzamos el objetivo de juegos
    detail_url = f"https://store.steampowered.com/api/appdetails?appids={app_id}&l=english"
    try: # Petición individual juego a juego con User-Agent
        response = requests.get(detail_url, headers = headers, timeout = 10)
        if response.status_code == 429: # Manejo de Rate Limit
            time.sleep(10)
            continue
        res = response.json()
        app_str = str(app_id)
        if res and app_str in res and res[app_str].get('success', False):
            data = res[app_str]['data']
            if data.get('type') != 'game': # Ignorar si es un DLC o Demo, solo queremos juegos base
                continue
            raw_date = data.get('release_date', {}).get('date', '').strip()
            parsed_date = pd.to_datetime(raw_date, errors='coerce') # Intentar parsear fecha estándar
            if pd.notna(parsed_date):
                year = int(parsed_date.year)
                month = int(parsed_date.month)
            else: # Si viene con formato "Coming soon", "Q4 2027", etc., extraemos el año con Regex
                years_found = re.findall(r'20\d{2}', raw_date)
                if years_found:
                    extracted_year = int(years_found[0]) # Si detecta 2026 (sin mes parseable) o años desorbitados (> 2030), lo pasamos a 2027
                    if extracted_year <= 2026 or extracted_year > 2030:
                        year = 2027
                    else:
                        year = extracted_year
                else: # Si no hay año en el texto ("Coming soon", "TBA"), asignamos 2027
                    year = 2027
                month = 1
            if (year == 2026 and month <= 7) or (year < 2026): # Omitir juegos de este año anteriores a julio y años pasados
                continue
            raw_languages = data.get('supported_languages', '') # Extraer idiomas (limpieza básica de etiquetas HTML)
            clean_languages = raw_languages.split('<br>')[0] if '<br>' in raw_languages else raw_languages
            languages_list = [lang.strip() for lang in clean_languages.split(',') if lang.strip()]
            num_generos = len(data.get('genres', [])) # Extraemos número de géneros
            price_info = data.get('price_overview') # Precio (Steam da los precios en céntimos; si no tiene o es F2P, marcamos 0.0)
            price = (price_info['final'] / 100.0) if price_info else 0.0
            game_entry = {
                'AppID': app_id,
                'Name': data.get('name'),
                'año_lanzamiento': year,
                'mes_lanzamiento': month,
                'lanzamiento_navidad': 1 if month in [11, 12] else 0,
                'Required age': data.get('required_age', 0),
                'Price': price,
                'Windows': data.get('platforms', {}).get('windows', False),
                'Mac': data.get('platforms', {}).get('mac', False),
                'Linux': data.get('platforms', {}).get('linux', False),
                'Developers': ", ".join(data.get('developers', [])),
                'Publishers': ", ".join(data.get('publishers', [])),
                'num_generos': num_generos,
                'num_idiomas': len(languages_list)
            }
            games_list.append(game_entry)
            if len(games_list) % 10 == 0:
                print(f"Juegos acumulados válidos: {len(games_list)} / {len(app_ids)}")
            time.sleep(1.5) # Pausa por juego para no saturar la API
    except Exception as e:
        time.sleep(1) # Pausa por juego para no saturar la API

df_proximos = pd.DataFrame(games_list)
print(f"Dataset creado con {len(df_proximos)} juegos.")
df_proximos.head()

Extrayendo y procesando características desde Steam...
Juegos acumulados válidos: 10 / 5536
Juegos acumulados válidos: 20 / 5536
Juegos acumulados válidos: 30 / 5536
Juegos acumulados válidos: 40 / 5536
Juegos acumulados válidos: 50 / 5536
Juegos acumulados válidos: 60 / 5536
Juegos acumulados válidos: 70 / 5536
Juegos acumulados válidos: 80 / 5536
Juegos acumulados válidos: 90 / 5536
Juegos acumulados válidos: 100 / 5536
Juegos acumulados válidos: 110 / 5536
Juegos acumulados válidos: 120 / 5536
Juegos acumulados válidos: 130 / 5536
Juegos acumulados válidos: 140 / 5536
Juegos acumulados válidos: 150 / 5536
Juegos acumulados válidos: 160 / 5536
Juegos acumulados válidos: 170 / 5536
Juegos acumulados válidos: 180 / 5536
Juegos acumulados válidos: 190 / 5536
Juegos acumulados válidos: 200 / 5536
Juegos acumulados válidos: 210 / 5536
Juegos acumulados válidos: 220 / 5536
Juegos acumulados válidos: 230 / 5536
Juegos acumulados válidos: 240 / 5536
Juegos acumulados válidos: 250 / 5536
Jueg

,AppID,Name,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,Required age,Price,Windows,Mac,Linux,Developers,Publishers,num_generos,num_idiomas
0,2001760,Beast of Reincarnation,2026,8,0,0,54.99,True,False,False,GAME FREAK inc.,Fictions,3,11
1,2950790,IRON NEST: Heavy Turret Simulator,2026,8,0,0,0.00,True,False,False,"Nick Nieuwoudt, Dominik Latos","Nick Nieuwoudt, Dominik Latos",3,16
2,4656000,BOMBANANA!,2026,8,0,0,0.00,True,True,False,Lefto Studio,TARK,2,15
3,1478500,Big Walk,2026,8,0,0,0.00,True,True,False,House House,Panic,1,14
4,4450620,GRAIN ROT,2026,8,0,0,0.00,True,False,False,Beck & Branch Games,Neem,3,11


In [46]:
df_proximos.to_csv("proximos_lanzamientos_steam.csv", index = False)

# Módulo de Limpieza, Validación y Feature Engineering del Dataset de Próximos Lanzamientos

## 1. Justificación y Objetivos
Durante la revisión del dataset de **1.000 juegos futuros** extraídos de la API de Steam, se detectó que el 97% de los registros presentaban un valor `Price = 0.0`. 

Tras un análisis exploratorio, se desestimó la imputación de precios arbitrarios (ej. 14.99 € por defecto a los indies), ya que rompía la variabilidad natural del dato y asumía precios que las editoras aún no han publicado. En su lugar, se optó por tratar la ausencia de precio no como un fallo, sino como una característica inherente al estado del ciclo de vida del producto antes de abrir la preventa.

---

## 2. Decisiones de Limpieza y Corrección de Anomalias

### A. Gestión de Duplicados
* **Problema:** Se identificaron entradas duplicadas con diferente `AppID` correspondientes a paquetes recopilatorios (*Bundles*) lanzados por la misma desarrolladora (ej. colecciones de *Konami*).
* **Solución:** Eliminación defensiva basada en la columna `Name` (`drop_duplicates(subset=['Name'])`), reduciendo el catálogo a **998 títulos únicos**.

### B. Tratamiento del Outlier de Precio
* **Problema:** El título *The Blood of Dawnwalker* registraba un precio erróneo de **289.00 €**.
* **Causa:** La API de Steam devolvió una cifra introducida temporalmente en el backend en una divisa no europea o con formato de prueba pre-lanzamiento.
* **Solución:** Corrección manual del valor a **69.99 €** (precio base oficial de venta).

### C. Acotación de Idiomas Localizados
* **Problema:** Títulos indies mostraban picos extremos de hasta **103 idiomas**, provocados por desarrolladores que marcan todas las traducciones automáticas disponibles en Steam.
* **Solución:** Aplicación de un límite superior (*Clipping*) a un máximo de **30 idiomas** para evitar sesgos de escala en modelos de regresión o árboles de decisión.

---

## 3. Estrategia de Feature Engineering y Homogeneización

Para permitir que el modelo analice la envergadura del proyecto sin depender de un precio no publicado, se crearon las siguientes variables derivadas:

1. **`Price` & `precio_anunciado`:** 
   * Los valores `0.0` se reemplazaron por `NaN` (valor ausente).
   * Se creó una variable binaria `precio_anunciado` ($1$ si la preventa está abierta con precio explícito; $0$ si el precio sigue pendiente).

2. **`es_autoeditado` ($0$ / $1$):**
   * Indicador binario ($1$ cuando `Developers == Publishers`). Aporta una métrica directa del perfil *indie* / *solo-dev* (representando el 74,4% de la muestra).

3. **`es_top_publisher` ($0$ / $1$):**
   * Identifica si el desarrollo cuenta con el respaldo de grandes editoras multinacionales (*Xbox Game Studios, Konami, Sega, Capcom, Bandai Namco, EA, etc.*).

4. **`num_plataformas` ($1$ a $3$):**
   * Suma lógica de los sistemas soportados (`Windows` + `Mac` + `Linux`), utilizada como *proxy* de la inversión técnica en el desarrollo.

---

## 4. Ajuste del Dataset Histórico

Para garantizar la **interoperabilidad y homogeneidad** durante el entrenamiento del modelo de Machine Learning, el dataset histórico de lanzamientos pasados fue procesado bajo las mismas transformaciones (`num_idiomas` acotado a 30, creación de `es_autoeditado`, `es_top_publisher` y `num_plataformas`). De este modo, la estructura matricial de características es idéntica tanto en la fase de entrenamiento como en la de inferencia futura.

In [22]:
# Cargamos el dataset de próximos lanzamientos para no tener que volver a ejecutar la llamda a Steam
df = pd.read_csv('proximos_lanzamientos_steam.csv')
df.head()

,AppID,Name,año_lanzamiento,mes_lanzamiento,lanzamiento_navidad,Required age,Price,Windows,Mac,Linux,Developers,Publishers,num_generos,num_idiomas
0,2001760,Beast of Reincarnation,2026,8,0,0,54.99,True,False,False,GAME FREAK inc.,Fictions,3,11
1,2950790,IRON NEST: Heavy Turret Simulator,2026,8,0,0,0.00,True,False,False,"Nick Nieuwoudt, Dominik Latos","Nick Nieuwoudt, Dominik Latos",3,16
2,4656000,BOMBANANA!,2026,8,0,0,0.00,True,True,False,Lefto Studio,TARK,2,15
3,1478500,Big Walk,2026,8,0,0,0.00,True,True,False,House House,Panic,1,14
4,4450620,GRAIN ROT,2026,8,0,0,0.00,True,False,False,Beck & Branch Games,Neem,3,11


In [ ]:
if 'AppID' in df.columns: # Eliminar AppID del dataset si está presente
    df = df.drop(columns = ['AppID'])

nuevos_juegos = [ # Añadimos esta lista de juegos a mano que no se encontraban en el dataset y son interesantes próximos lanzamientos
    {'Name' : 'Beyond Good & Evil 2', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Ubisoft', 'Publishers': 'Ubisoft', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'Grand Theft Auto VI', 'año_lanzamiento' : 2026, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Rockstar Games', 'Publishers': 'Take-Two Interactive', 'num_generos': 3, 'num_idiomas': 13},
    {'Name' : 'Little Devil Inside', 'año_lanzamiento' : 2026, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 0, 'Price': 29.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Neostream Interactive', 'Publishers': 'Neostream Interactive', 'num_generos': 3, 'num_idiomas': 8},
    {'Name' : 'The Elder Scrolls VI', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Bethesda Game Studios', 'Publishers': 'Xbox Game Studios', 'num_generos': 3, 'num_idiomas': 14},
    {'Name' : 'Mass Effect 5', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Bioware', 'Publishers': 'Electronic Arts', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'The Witcher 4', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Cd Projekt Red', 'Publishers': 'Cd Projekt Red', 'num_generos': 3, 'num_idiomas': 15},
    {'Name' : 'The Last Night', 'año_lanzamiento' : 2026, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 0, 'Price': 24.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Odd Tales', 'Publishers': 'Raw Fury', 'num_generos': 3, 'num_idiomas': 10},
    {'Name' : 'Stranger Than Heaven', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Ryu Ga Gotoku Studio', 'Publishers': 'SEGA', 'num_generos': 3, 'num_idiomas': 13},
    {'Name' : 'Metro 2039', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 59.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': '4A Games', 'Publishers': 'Deep Silver', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'Kingdom Hearts 4', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 9, 'lanzamiento_navidad': 0, 'Required age': 12, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Square Enix', 'Publishers': 'Square Enix', 'num_generos': 3, 'num_idiomas': 10},
    {'Name' : 'Bioshock 4', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Cloud Chamber', 'Publishers': '2K', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'Star Wars Eclipse', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 16, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Quantic Dream', 'Publishers': 'Lucasfilm Games', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'Exodus', 'año_lanzamiento' : 2026, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 59.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Archetype Entertainment', 'Publishers': 'Wizards of the Coast', 'num_generos': 3, 'num_idiomas': 10},
    {'Name' : "Assassin's Creed Codename HEXE", 'año_lanzamiento' : 2027, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 18, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Ubisoft Montreal', 'Publishers': 'Ubisoft', 'num_generos': 3, 'num_idiomas': 14},
    {'Name' : 'Divinity', 'año_lanzamiento': 2027, 'mes_lanzamiento' : 10, 'lanzamiento_navidad': 0, 'Required age': 18, 'Price': 59.99, 'Windows': 1, 'Mac': 0 'Linux': 0, 'Developers': 'Larian Studios', 'Publishers': 'Larian Studios', 'num_generos': 3, 'num_idiomas': 13},
    {'Name' : 'Star Wars Fate of the Old Republic', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 11, 'lanzamiento_navidad': 1, 'Required age': 16, 'Price': 69.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'BioWare', 'Publishers': 'Electronic Arts', 'num_generos': 3, 'num_idiomas': 11},
    {'Name' : 'DokeV', 'año_lanzamiento': 2026, 'mes_lanzamiento' : 11, 'lanzamiento_navidad': 1, 'Required age': 0, 'Price': 39.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Pearl Abyss', 'Publishers': 'Pearl Abyss', 'num_generos': 3, 'num_idiomas': 12},
    {'Name' : 'In the Valley of Gods', 'año_lanzamiento' : 2027, 'mes_lanzamiento': 10, 'lanzamiento_navidad': 0, 'Required age': 0, 'Price': 29.99, 'Windows': 1, 'Mac': 0, 'Linux': 0, 'Developers': 'Campo Santo', 'Publishers': 'Valve', 'num_generos': 2, 'num_idiomas': 10}
]